# B-Scan Migration Playground — Subwavelength FluidFlow TimeLapse Analysis

**Purpose:** Quantify the lateral movement detection limit of a moving fluid front through
a subwavelength thick fracture using gprMax simulations.

The fracture hosts a **Graded Transitional Wetting Zone**: a capillary-fringe-style
permittivity ramp from water (εr=80) down to air (εr=1) in steps of 10, rather than a
sharp two-material boundary. Each intermediate step (εr = 70, 60, 50, 40, 30, 20, 10) is
its own box, 1/10 of a wavelength in ice wide. The whole graded assembly travels laterally
through the same 8 scenarios as the original sharp-front study.

## Geometry

## Migration methods (all post-stack / zero-offset)
| Method | Domain | Velocity use |
|--------|--------|---------------|

## Sensitivity sweep
The fluid front moves w.r.t. a baseline study to create the timelapsed study. The fluid front moves laterally by a fraction of the wavelength in the medium (2 lambda to 1/32 lambda). After migration the sensitivity of is determined if this movement can be detected. Both migration before and after taking the timelapse difference is tested.


In [ ]:
import os, time as _time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

import os
import numpy as np
import pyvista as pv
import matplotlib.pyplot as plt
from gprMax.gprMax import api
from tools.outputfiles_merge import merge_files
from tools.plot_Bscan import get_output_data, mpl_plot


In [ ]:
# ── Standard figure auto-save (protocol: .wiki/FIGURES_PROTOCOL.md) ─────────
import sys, pathlib
if str(pathlib.Path.cwd()) not in sys.path:
    sys.path.insert(0, str(pathlib.Path.cwd()))
from helper_functions.figures import setup_autosave
setup_autosave(study="FluidFlow_Study", prefix="FF_")


In [ ]:
# Medium
eps_r_ice      = 3.15 # ice
eps_r_air      = 1.0  # air
eps_r_water    = 80.0 # water

v_ice      = 0.299792458 / np.sqrt(eps_r_ice)   # m/ns ≈ 0.16892
v_water    = 0.299792458 / np.sqrt(eps_r_water) # m/ns ≈ 0.03357
f_c_GHz    = 1.5                              # centre frequency [GHz]
wavelength_ice = v_ice / f_c_GHz                  # m ≈ 0.1126
wavelength_water = v_water / f_c_GHz          # m ≈ 0.02238
v_mig      = v_ice / 2                        # exploding-reflector half-velocity [m/ns]
t0_ns      = np.sqrt(2) / f_c_GHz            # Ricker peak delay [ns] ≈ 0.943

# Survey geometry
domain_x   = 4.0
n_traces   = 380
trace_step = 0.01   # m
rx_offset  = 0.1    # m (source-receiver offset in .in file)
x_traces   = (0.1 + rx_offset / 2) + np.arange(n_traces) * trace_step   # midpoints [m]

# Scatterer geometry (shared across all datasets)
x_centre        = domain_x / 2
y_surface       = 0.9                          # air-ice interface [m from bottom]
y_scatterer     = y_surface - wavelength_ice * 6  # cylinder centre [m from bottom]
z_scatterer     = y_surface - y_scatterer     # depth below surface [m] ≈ 0.676
fracture_thickness     = wavelength_ice / 40             # radius of cylinder (point scatterer approximation) [m] ≈ 0.0028
z_top           = z_scatterer - fracture_thickness   # depth to cylinder top (first reflection)

shift = np.array([2, 1, 0.5, 0.25, 0.125, 0.0625, 0.03125]) * wavelength_ice
labels      = ['Baseline', '2λ', '1λ', '½λ', '¼λ', '⅛λ', '¹⁄₁₆λ', '¹⁄₃₂λ']

# Migration depth grid
z_img = np.linspace(0.0, 0.8, 160)

# Paths
STUDY_ROOT = Path(r'C:\Users\Administrator\OneDrive\Thesis\TimeLapse_Notebooks\fluidflow_study')

print(f'λ_ice       = {wavelength_ice*1e3:.1f} mm')
print(f'λ_water     = {wavelength_water*1e3:.1f} mm')
print(f'v_ice       = {v_ice:.5f} m/ns,  v_mig = {v_mig:.5f} m/ns')
print(f't0_ns       = {t0_ns:.3f} ns')
print(f'z_scatterer = {z_scatterer:.3f} m  (centre),  z_top = {z_top:.4f} m')
print(f'x_traces    : {x_traces[0]:.3f} → {x_traces[-1]:.3f} m  ({n_traces} traces)')


# Input File Creation

## Calculate Geometry Factors

In [ ]:
separation = np.array([0, 2, 1, 1/2, 1/4, 1/8, 1/16, 1/32]) * wavelength_ice  # m

x_centre = domain_x / 2
x_scatterer_1 = np.round(x_centre - wavelength_ice * 3 + separation, 3) # 8 locations, wetting-zone centre

# check if discretisation is sufficient for CFL
dx_required = (v_water / 4.5) / 10 # 4 GHz is the highest frequency component in the Ricker wavelet
if 0.001 < dx_required:
    print(f"Discretisation is sufficiently small")
else:
    print(f"Discretisation is too large for stability (dx should be < {dx_required:.4f} m)")
if fracture_thickness < dx_required:
    print(f"Scatterer radius is smaller than discretisation")
else:
    print(f"Scatterer radius is sufficiently large for stability (should be > {dx_required:.4f} m)")


## Graded Transitional Wetting Zone Geometry

Instead of a sharp water/air edge, the fracture hosts a graded permittivity ramp:
water (εr=80) → 70 → 60 → 50 → 40 → 30 → 20 → 10 → air (εr=1). Each of the 7
intermediate materials gets its own box, exactly `wavelength_ice/10` wide. The whole
9-box assembly is centred on `x_scatterer_1[i]` for each scenario (so that value now
represents the centre of the graded transition, not a sharp edge).

**Modelling assumption:** conductivity is assumed to scale linearly with permittivity
between water (σ=0.01 S/m) and air (σ≈0 S/m), i.e. `sigma(eps) = 0.01 * eps/80` — a
simplifying choice, not a literature mixing law (e.g. CRIM/Bruggeman).


In [ ]:
eps_steps   = np.array([70, 60, 50, 40, 30, 20, 10])         # 7 intermediate materials
sigma_steps = 0.01 * eps_steps / 80                           # linear-in-eps assumption (documented above)
box_width   = wavelength_ice / 10                             # ≈ 0.011261 m
total_width = box_width * len(eps_steps)                      # ≈ 0.078826 m
half_width  = total_width / 2                                 # ≈ 0.039413 m

def partition_edges(start_x, total_width, n_boxes):
    # Gap-free, monotonic partition of [start_x, start_x+total_width] into n_boxes
    # equal sub-intervals, each edge independently rounded to gprMax's dx=0.001 grid.
    raw_edges = start_x + np.linspace(0, total_width, n_boxes + 1)
    edges = np.round(raw_edges, 3)
    for i in range(1, len(edges)):
        if edges[i] <= edges[i - 1]:
            edges[i] = edges[i - 1] + 0.001       # defensive; not expected to trigger
    return edges

edges_all = [partition_edges(x - half_width, total_width, len(eps_steps)) for x in x_scatterer_1]

print(f'box_width = {box_width*1e3:.2f} mm, total transition width = {total_width*1e3:.2f} mm')
print('\nPer-scenario graded-zone edges (water | 70 | 60 | 50 | 40 | 30 | 20 | 10 | air):')
for lbl, edges in zip(labels, edges_all):
    widths = np.diff(edges)
    flag = '  <-- WARNING: non-positive width!' if (widths <= 0).any() else ''
    print(f'  {lbl:>8}: centre={ (edges[0]+edges[-1])/2 :.3f} m  edges=' +
          ','.join(f'{e:.3f}' for e in edges) + f'{flag}')


## Background model (no scatterers)

In [ ]:
%%writefile fluidflow_study/background/background.in

#title: GPR Fluidflow Study - background
#domain: 4.000 1.000 0.001
#dx_dy_dz: 0.001 0.001 0.001
#time_window: 20e-9
#pml_cells: 10 10 0 10 10 0

#material: 3.15 1e-6 1.0 0 ice

// Subsurface background model
#box: 0 0 0 4.000 0.900 0.001 ice

#python:
from gprMax.input_cmd_funcs import *

# current_model_run starts at 1 and goes up to the -n value
iteration = current_model_run - 1  # Offset to start at 0

# Calculate current x-position
curr_x = 0.100 + (iteration * 0.01)

# Place exactly ONE source and ONE receiver
waveform('ricker', 1.0, 1.5e9, 'my_ricker')
hertzian_dipole('z', curr_x, 0.900, 0, 'my_ricker')
rx(curr_x + 0.1, 0.900, 0)
#end_python:



#geometry_view: 0 0 0 4.000 1.000 0.001 0.001 0.001 0.001 background n
#messages: y


# Baseline (graded wetting zone)

In [ ]:
%%writefile fluidflow_study/baseline/resolution_baseline.in

#title: GPR Fluidflow Study - baseline (Graded Transitional Wetting Zone)
#domain: 4.000 1.000 0.001
#dx_dy_dz: 0.001 0.001 0.001
#time_window: 20e-9
#pml_cells: 10 10 0 10 10 0

#material: 3.15 1e-6 1.0 0 ice
#material: 1 0 1 0 air
#material: 80 0.01 1 0 water
#material: 70 0.00875 1 0 wet_zone_70
#material: 60 0.0075  1 0 wet_zone_60
#material: 50 0.00625 1 0 wet_zone_50
#material: 40 0.005   1 0 wet_zone_40
#material: 30 0.00375 1 0 wet_zone_30
#material: 20 0.0025  1 0 wet_zone_20
#material: 10 0.00125 1 0 wet_zone_10

// Subsurface background model
#box: 0 0 0 4.000 0.900 0.001 ice

// Graded transitional wetting zone (water -> 7 graded steps -> air)
#box: 0 0.2212 0 1.623 0.224 0.001 water
#box: 1.623 0.2212 0 1.634 0.224 0.001 wet_zone_70
#box: 1.634 0.2212 0 1.645 0.224 0.001 wet_zone_60
#box: 1.645 0.2212 0 1.656 0.224 0.001 wet_zone_50
#box: 1.656 0.2212 0 1.668 0.224 0.001 wet_zone_40
#box: 1.668 0.2212 0 1.679 0.224 0.001 wet_zone_30
#box: 1.679 0.2212 0 1.690 0.224 0.001 wet_zone_20
#box: 1.690 0.2212 0 1.701 0.224 0.001 wet_zone_10
#box: 1.701 0.2212 0 4.000 0.224 0.001 air

#python:
from gprMax.input_cmd_funcs import *

# current_model_run starts at 1 and goes up to the -n value
iteration = current_model_run - 1  # Offset to start at 0

# Calculate current x-position
curr_x = 0.100 + (iteration * 0.01)

# Place exactly ONE source and ONE receiver
waveform('ricker', 1.0, 1.5e9, 'my_ricker')
hertzian_dipole('z', curr_x, 0.900, 0, 'my_ricker')
rx(curr_x + 0.1, 0.900, 0)
#end_python:


#geometry_view: 0 0 0 4.000 1.000 0.001 0.001 0.001 0.001 baseline n
#messages: y


## 2 $\lambda$ lateral shift

In [ ]:
%%writefile fluidflow_study/shift_2lambda/resolution_2lambda.in

#title: GPR Fluidflow Study - 2 wavelength shift
#domain: 4.000 1.000 0.001
#dx_dy_dz: 0.001 0.001 0.001
#time_window: 20e-9
#pml_cells: 10 10 0 10 10 0

#material: 3.15 1e-6 1.0 0 ice
#material: 1 0 1 0 air
#material: 80 0.01 1 0 water
#material: 70 0.00875 1 0 wet_zone_70
#material: 60 0.0075  1 0 wet_zone_60
#material: 50 0.00625 1 0 wet_zone_50
#material: 40 0.005   1 0 wet_zone_40
#material: 30 0.00375 1 0 wet_zone_30
#material: 20 0.0025  1 0 wet_zone_20
#material: 10 0.00125 1 0 wet_zone_10

// Subsurface background model
#box: 0 0 0 4.000 0.900 0.001 ice

// Graded transitional wetting zone (water -> 7 graded steps -> air)
#box: 0 0.2212 0 1.848 0.224 0.001 water
#box: 1.848 0.2212 0 1.859 0.224 0.001 wet_zone_70
#box: 1.859 0.2212 0 1.870 0.224 0.001 wet_zone_60
#box: 1.870 0.2212 0 1.881 0.224 0.001 wet_zone_50
#box: 1.881 0.2212 0 1.893 0.224 0.001 wet_zone_40
#box: 1.893 0.2212 0 1.904 0.224 0.001 wet_zone_30
#box: 1.904 0.2212 0 1.915 0.224 0.001 wet_zone_20
#box: 1.915 0.2212 0 1.926 0.224 0.001 wet_zone_10
#box: 1.926 0.2212 0 4.000 0.224 0.001 air

#python:
from gprMax.input_cmd_funcs import *

# current_model_run starts at 1 and goes up to the -n value
iteration = current_model_run - 1  # Offset to start at 0

# Calculate current x-position
curr_x = 0.100 + (iteration * 0.01)

# Place exactly ONE source and ONE receiver
waveform('ricker', 1.0, 1.5e9, 'my_ricker')
hertzian_dipole('z', curr_x, 0.900, 0, 'my_ricker')
rx(curr_x + 0.1, 0.900, 0)
#end_python:


#geometry_view: 0 0 0 4.000 1.000 0.001 0.001 0.001 0.001 fluid_front_2_lambda n
#messages: y


## 1 $\lambda$ lateral shift

In [ ]:
%%writefile fluidflow_study/shift_1lambda/resolution_1lambda.in

#title: GPR Fluidflow Study - 1 wavelength shift
#domain: 4.000 1.000 0.001
#dx_dy_dz: 0.001 0.001 0.001
#time_window: 20e-9
#pml_cells: 10 10 0 10 10 0

#material: 3.15 1e-6 1.0 0 ice
#material: 1 0 1 0 air
#material: 80 0.01 1 0 water
#material: 70 0.00875 1 0 wet_zone_70
#material: 60 0.0075  1 0 wet_zone_60
#material: 50 0.00625 1 0 wet_zone_50
#material: 40 0.005   1 0 wet_zone_40
#material: 30 0.00375 1 0 wet_zone_30
#material: 20 0.0025  1 0 wet_zone_20
#material: 10 0.00125 1 0 wet_zone_10

// Subsurface background model
#box: 0 0 0 4.000 0.900 0.001 ice

// Graded transitional wetting zone (water -> 7 graded steps -> air)
#box: 0 0.2212 0 1.736 0.224 0.001 water
#box: 1.736 0.2212 0 1.747 0.224 0.001 wet_zone_70
#box: 1.747 0.2212 0 1.758 0.224 0.001 wet_zone_60
#box: 1.758 0.2212 0 1.769 0.224 0.001 wet_zone_50
#box: 1.769 0.2212 0 1.781 0.224 0.001 wet_zone_40
#box: 1.781 0.2212 0 1.792 0.224 0.001 wet_zone_30
#box: 1.792 0.2212 0 1.803 0.224 0.001 wet_zone_20
#box: 1.803 0.2212 0 1.814 0.224 0.001 wet_zone_10
#box: 1.814 0.2212 0 4.000 0.224 0.001 air

#python:
from gprMax.input_cmd_funcs import *

# current_model_run starts at 1 and goes up to the -n value
iteration = current_model_run - 1  # Offset to start at 0

# Calculate current x-position
curr_x = 0.100 + (iteration * 0.01)

# Place exactly ONE source and ONE receiver
waveform('ricker', 1.0, 1.5e9, 'my_ricker')
hertzian_dipole('z', curr_x, 0.900, 0, 'my_ricker')
rx(curr_x + 0.1, 0.900, 0)
#end_python:


#geometry_view: 0 0 0 4.000 1.000 0.001 0.001 0.001 0.001 fluid_front_1_lambda n
#messages: y


## 1/2 $\lambda$ lateral shift

In [ ]:
%%writefile fluidflow_study/shift_0p5lambda/resolution_0p5lambda.in

#title: GPR Fluidflow Study - 0.5 wavelength shift
#domain: 4.000 1.000 0.001
#dx_dy_dz: 0.001 0.001 0.001
#time_window: 20e-9
#pml_cells: 10 10 0 10 10 0

#material: 3.15 1e-6 1.0 0 ice
#material: 1 0 1 0 air
#material: 80 0.01 1 0 water
#material: 70 0.00875 1 0 wet_zone_70
#material: 60 0.0075  1 0 wet_zone_60
#material: 50 0.00625 1 0 wet_zone_50
#material: 40 0.005   1 0 wet_zone_40
#material: 30 0.00375 1 0 wet_zone_30
#material: 20 0.0025  1 0 wet_zone_20
#material: 10 0.00125 1 0 wet_zone_10

// Subsurface background model
#box: 0 0 0 4.000 0.900 0.001 ice

// Graded transitional wetting zone (water -> 7 graded steps -> air)
#box: 0 0.2212 0 1.679 0.224 0.001 water
#box: 1.679 0.2212 0 1.690 0.224 0.001 wet_zone_70
#box: 1.690 0.2212 0 1.701 0.224 0.001 wet_zone_60
#box: 1.701 0.2212 0 1.712 0.224 0.001 wet_zone_50
#box: 1.712 0.2212 0 1.724 0.224 0.001 wet_zone_40
#box: 1.724 0.2212 0 1.735 0.224 0.001 wet_zone_30
#box: 1.735 0.2212 0 1.746 0.224 0.001 wet_zone_20
#box: 1.746 0.2212 0 1.757 0.224 0.001 wet_zone_10
#box: 1.757 0.2212 0 4.000 0.224 0.001 air

#python:
from gprMax.input_cmd_funcs import *

# current_model_run starts at 1 and goes up to the -n value
iteration = current_model_run - 1  # Offset to start at 0

# Calculate current x-position
curr_x = 0.100 + (iteration * 0.01)

# Place exactly ONE source and ONE receiver
waveform('ricker', 1.0, 1.5e9, 'my_ricker')
hertzian_dipole('z', curr_x, 0.900, 0, 'my_ricker')
rx(curr_x + 0.1, 0.900, 0)
#end_python:


#geometry_view: 0 0 0 4.000 1.000 0.001 0.001 0.001 0.001 fluid_front_0p5_lambda n
#messages: y


## 1/4 $\lambda$ lateral shift

In [ ]:
%%writefile fluidflow_study/shift_0p25lambda/resolution_0p25lambda.in

#title: GPR Fluidflow Study - 0.25 wavelength shift
#domain: 4.000 1.000 0.001
#dx_dy_dz: 0.001 0.001 0.001
#time_window: 20e-9
#pml_cells: 10 10 0 10 10 0

#material: 3.15 1e-6 1.0 0 ice
#material: 1 0 1 0 air
#material: 80 0.01 1 0 water
#material: 70 0.00875 1 0 wet_zone_70
#material: 60 0.0075  1 0 wet_zone_60
#material: 50 0.00625 1 0 wet_zone_50
#material: 40 0.005   1 0 wet_zone_40
#material: 30 0.00375 1 0 wet_zone_30
#material: 20 0.0025  1 0 wet_zone_20
#material: 10 0.00125 1 0 wet_zone_10

// Subsurface background model
#box: 0 0 0 4.000 0.900 0.001 ice

// Graded transitional wetting zone (water -> 7 graded steps -> air)
#box: 0 0.2212 0 1.651 0.224 0.001 water
#box: 1.651 0.2212 0 1.662 0.224 0.001 wet_zone_70
#box: 1.662 0.2212 0 1.673 0.224 0.001 wet_zone_60
#box: 1.673 0.2212 0 1.684 0.224 0.001 wet_zone_50
#box: 1.684 0.2212 0 1.696 0.224 0.001 wet_zone_40
#box: 1.696 0.2212 0 1.707 0.224 0.001 wet_zone_30
#box: 1.707 0.2212 0 1.718 0.224 0.001 wet_zone_20
#box: 1.718 0.2212 0 1.729 0.224 0.001 wet_zone_10
#box: 1.729 0.2212 0 4.000 0.224 0.001 air

#python:
from gprMax.input_cmd_funcs import *

# current_model_run starts at 1 and goes up to the -n value
iteration = current_model_run - 1  # Offset to start at 0

# Calculate current x-position
curr_x = 0.100 + (iteration * 0.01)

# Place exactly ONE source and ONE receiver
waveform('ricker', 1.0, 1.5e9, 'my_ricker')
hertzian_dipole('z', curr_x, 0.900, 0, 'my_ricker')
rx(curr_x + 0.1, 0.900, 0)
#end_python:


#geometry_view: 0 0 0 4.000 1.000 0.001 0.001 0.001 0.001 fluid_front_0p25_lambda n
#messages: y


## 1/8 $\lambda$ lateral shift

In [ ]:
%%writefile fluidflow_study/shift_0p125lambda/resolution_0p125lambda.in

#title: GPR Fluidflow Study - 0.125 wavelength shift
#domain: 4.000 1.000 0.001
#dx_dy_dz: 0.001 0.001 0.001
#time_window: 20e-9
#pml_cells: 10 10 0 10 10 0

#material: 3.15 1e-6 1.0 0 ice
#material: 1 0 1 0 air
#material: 80 0.01 1 0 water
#material: 70 0.00875 1 0 wet_zone_70
#material: 60 0.0075  1 0 wet_zone_60
#material: 50 0.00625 1 0 wet_zone_50
#material: 40 0.005   1 0 wet_zone_40
#material: 30 0.00375 1 0 wet_zone_30
#material: 20 0.0025  1 0 wet_zone_20
#material: 10 0.00125 1 0 wet_zone_10

// Subsurface background model
#box: 0 0 0 4.000 0.900 0.001 ice

// Graded transitional wetting zone (water -> 7 graded steps -> air)
#box: 0 0.2212 0 1.637 0.224 0.001 water
#box: 1.637 0.2212 0 1.648 0.224 0.001 wet_zone_70
#box: 1.648 0.2212 0 1.659 0.224 0.001 wet_zone_60
#box: 1.659 0.2212 0 1.670 0.224 0.001 wet_zone_50
#box: 1.670 0.2212 0 1.682 0.224 0.001 wet_zone_40
#box: 1.682 0.2212 0 1.693 0.224 0.001 wet_zone_30
#box: 1.693 0.2212 0 1.704 0.224 0.001 wet_zone_20
#box: 1.704 0.2212 0 1.715 0.224 0.001 wet_zone_10
#box: 1.715 0.2212 0 4.000 0.224 0.001 air

#python:
from gprMax.input_cmd_funcs import *

# current_model_run starts at 1 and goes up to the -n value
iteration = current_model_run - 1  # Offset to start at 0

# Calculate current x-position
curr_x = 0.100 + (iteration * 0.01)

# Place exactly ONE source and ONE receiver
waveform('ricker', 1.0, 1.5e9, 'my_ricker')
hertzian_dipole('z', curr_x, 0.900, 0, 'my_ricker')
rx(curr_x + 0.1, 0.900, 0)
#end_python:


#geometry_view: 0 0 0 4.000 1.000 0.001 0.001 0.001 0.001 fluid_front_0p125_lambda n
#messages: y


## 1/16 $\lambda$ lateral shift

In [ ]:
%%writefile fluidflow_study/shift_0p0625lambda/resolution_0p0625lambda.in

#title: GPR Fluidflow Study - 0.0625 wavelength shift
#domain: 4.000 1.000 0.001
#dx_dy_dz: 0.001 0.001 0.001
#time_window: 20e-9
#pml_cells: 10 10 0 10 10 0

#material: 3.15 1e-6 1.0 0 ice
#material: 1 0 1 0 air
#material: 80 0.01 1 0 water
#material: 70 0.00875 1 0 wet_zone_70
#material: 60 0.0075  1 0 wet_zone_60
#material: 50 0.00625 1 0 wet_zone_50
#material: 40 0.005   1 0 wet_zone_40
#material: 30 0.00375 1 0 wet_zone_30
#material: 20 0.0025  1 0 wet_zone_20
#material: 10 0.00125 1 0 wet_zone_10

// Subsurface background model
#box: 0 0 0 4.000 0.900 0.001 ice

// Graded transitional wetting zone (water -> 7 graded steps -> air)
#box: 0 0.2212 0 1.630 0.224 0.001 water
#box: 1.630 0.2212 0 1.641 0.224 0.001 wet_zone_70
#box: 1.641 0.2212 0 1.652 0.224 0.001 wet_zone_60
#box: 1.652 0.2212 0 1.663 0.224 0.001 wet_zone_50
#box: 1.663 0.2212 0 1.675 0.224 0.001 wet_zone_40
#box: 1.675 0.2212 0 1.686 0.224 0.001 wet_zone_30
#box: 1.686 0.2212 0 1.697 0.224 0.001 wet_zone_20
#box: 1.697 0.2212 0 1.708 0.224 0.001 wet_zone_10
#box: 1.708 0.2212 0 4.000 0.224 0.001 air

#python:
from gprMax.input_cmd_funcs import *

# current_model_run starts at 1 and goes up to the -n value
iteration = current_model_run - 1  # Offset to start at 0

# Calculate current x-position
curr_x = 0.100 + (iteration * 0.01)

# Place exactly ONE source and ONE receiver
waveform('ricker', 1.0, 1.5e9, 'my_ricker')
hertzian_dipole('z', curr_x, 0.900, 0, 'my_ricker')
rx(curr_x + 0.1, 0.900, 0)
#end_python:


#geometry_view: 0 0 0 4.000 1.000 0.001 0.001 0.001 0.001 fluid_front_0p0625_lambda n
#messages: y


## 1/32 $\lambda$ lateral shift

In [ ]:
%%writefile fluidflow_study/shift_0p03125lambda/resolution_0p03125lambda.in

#title: GPR Fluidflow Study - 0.03125 wavelength shift
#domain: 4.000 1.000 0.001
#dx_dy_dz: 0.001 0.001 0.001
#time_window: 20e-9
#pml_cells: 10 10 0 10 10 0

#material: 3.15 1e-6 1.0 0 ice
#material: 1 0 1 0 air
#material: 80 0.01 1 0 water
#material: 70 0.00875 1 0 wet_zone_70
#material: 60 0.0075  1 0 wet_zone_60
#material: 50 0.00625 1 0 wet_zone_50
#material: 40 0.005   1 0 wet_zone_40
#material: 30 0.00375 1 0 wet_zone_30
#material: 20 0.0025  1 0 wet_zone_20
#material: 10 0.00125 1 0 wet_zone_10

// Subsurface background model
#box: 0 0 0 4.000 0.900 0.001 ice

// Graded transitional wetting zone (water -> 7 graded steps -> air)
#box: 0 0.2212 0 1.627 0.224 0.001 water
#box: 1.627 0.2212 0 1.638 0.224 0.001 wet_zone_70
#box: 1.638 0.2212 0 1.649 0.224 0.001 wet_zone_60
#box: 1.649 0.2212 0 1.660 0.224 0.001 wet_zone_50
#box: 1.660 0.2212 0 1.672 0.224 0.001 wet_zone_40
#box: 1.672 0.2212 0 1.683 0.224 0.001 wet_zone_30
#box: 1.683 0.2212 0 1.694 0.224 0.001 wet_zone_20
#box: 1.694 0.2212 0 1.705 0.224 0.001 wet_zone_10
#box: 1.705 0.2212 0 4.000 0.224 0.001 air

#python:
from gprMax.input_cmd_funcs import *

# current_model_run starts at 1 and goes up to the -n value
iteration = current_model_run - 1  # Offset to start at 0

# Calculate current x-position
curr_x = 0.100 + (iteration * 0.01)

# Place exactly ONE source and ONE receiver
waveform('ricker', 1.0, 1.5e9, 'my_ricker')
hertzian_dipole('z', curr_x, 0.900, 0, 'my_ricker')
rx(curr_x + 0.1, 0.900, 0)
#end_python:


#geometry_view: 0 0 0 4.000 1.000 0.001 0.001 0.001 0.001 fluid_front_0p03125_lambda n
#messages: y


# Visualising Model Geometry

In [ ]:
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt

# ── Geometry constants (from input files) ─────────────────────────────────────
domain_y = 1.0
dx_grid  = 0.001
pml_t    = 10 * dx_grid   # = 0.010 m
air_h    = domain_y - y_surface   # = 0.1 m
rx_off   = 0.1            # m  Tx-Rx offset

# Depth coordinate (0 = ice surface, positive = deeper)
d_top  = -air_h          # −0.1 m  top of domain
d_scat =  z_scatterer    #  0.676 m fracture depth

cmap_grade = plt.cm.Blues
def _grade_color(eps):
    return cmap_grade(0.15 + 0.75 * eps / 80.0)

# ── Figure 1: full domain cross-section (baseline graded zone, true scale) ───
fig1, ax1 = plt.subplots(figsize=(14, 4))

# Material regions
ax1.add_patch(Rectangle((0, 0), domain_x, y_surface,
                         facecolor='#f5f5f5', edgecolor='none'))   # ice (background)
ax1.add_patch(Rectangle((0, d_top), domain_x, air_h,
                         facecolor='#f2faff', edgecolor='none'))   # air above surface

# PML (left, right, top)
_p = dict(facecolor='#d0d0d0', edgecolor='#888', hatch='///', alpha=0.80, lw=0.4)
ax1.add_patch(Rectangle((0,               d_top), pml_t,   domain_y, **_p))
ax1.add_patch(Rectangle((domain_x - pml_t, d_top), pml_t,   domain_y, **_p))
ax1.add_patch(Rectangle((0,               d_top), domain_x, pml_t,   **_p))

ax1.add_patch(Rectangle((0, d_top), domain_x, domain_y,
                         fill=False, edgecolor='black', lw=1.5))
ax1.axhline(0, color='steelblue', lw=1.0, ls='--', alpha=0.7)

# Tx / Rx markers (every 5th trace)
x_src_arr = 0.100 + np.arange(n_traces) * trace_step
ax1.scatter(x_src_arr[::5],          np.zeros_like(x_src_arr[::5]),
            marker='^', s=10, color='navy',       zorder=6, label='Tx')
ax1.scatter(x_src_arr[::5] + rx_off, np.zeros_like(x_src_arr[::5]),
            marker='v', s=10, color='darkorange', zorder=6, label='Rx')

# Baseline graded wetting zone, drawn at true scale across the full domain width
fracture_y0, fracture_y1 = d_scat - fracture_thickness / 2, d_scat + fracture_thickness / 2
edges_base = edges_all[0]
ax1.add_patch(Rectangle((0, fracture_y0), edges_base[0], fracture_thickness,
                         facecolor='steelblue', edgecolor='none', zorder=5))
for k, eps in enumerate(eps_steps):
    ax1.add_patch(Rectangle((edges_base[k], fracture_y0), edges_base[k+1] - edges_base[k],
                             fracture_thickness, facecolor=_grade_color(eps), edgecolor='none', zorder=5))
ax1.add_patch(Rectangle((edges_base[-1], fracture_y0), domain_x - edges_base[-1], fracture_thickness,
                         facecolor='white', edgecolor='#aaa', lw=0.3, zorder=5))

ax1.axvline(x_scatterer_1[0], color='green', lw=1.0, ls='--', alpha=0.8, zorder=6)
ax1.annotate('baseline front centre\n(zoomed in Fig. 2)', xy=(x_scatterer_1[0], d_scat - 0.06),
             ha='center', va='top', fontsize=8, color='green')

# Region text
ax1.text(domain_x/2, d_top + air_h/2, 'Air  (ε_r = 1)',
         ha='center', va='center', fontsize=10, color='#444')
ax1.text(domain_x/2, y_surface/2, f'Ice  (ε_r = {eps_r_ice})',
         ha='center', va='center', fontsize=10, color='#1a4a6e')
ax1.text(pml_t/2,             d_top + domain_y/2, 'PML',
         ha='center', va='center', fontsize=8, color='#555', rotation=90)
ax1.text(domain_x - pml_t/2, d_top + domain_y/2, 'PML',
         ha='center', va='center', fontsize=8, color='#555', rotation=90)

ax1.set_xlim(0, domain_x)
ax1.set_ylim(y_surface, d_top)
ax1.set_xlabel('x [m]', fontsize=11)
ax1.set_ylabel('Depth [m]', fontsize=11)
ax1.set_title(
    f'FluidFlow Study — Model Geometry  '
    f'(domain {domain_x:.1f}×{domain_y:.1f} m, Δx = {dx_grid*1e3:.0f} mm, '
    f'PML = {pml_t*1e3:.0f} mm, f_c = {f_c_GHz} GHz, λ = {wavelength_ice*1e3:.1f} mm)  '
    f'— baseline graded wetting zone shown at true scale (water -> 7 steps -> air)',
    fontsize=10
)
ax1.set_aspect('equal', adjustable='box')
ax1.legend(loc='upper right', fontsize=9, ncol=2)
ax1.grid(True, ls='--', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ── Figure 2: zoomed — graded wetting zone at all 8 scenario positions ────────
# All 8 scenarios share the same depth (only x shifts), and the finest shifts are
# only ~3 mm apart — too close to tell apart if drawn at one shared depth. Instead,
# stack one horizontal row per scenario (y-axis = scenario, not physical depth) so
# every scenario's water/7-grade/air boxes are individually legible.

x_lo = min(e[0] for e in edges_all) - 0.01
x_hi = max(e[-1] for e in edges_all) + 0.01

fig2, ax2 = plt.subplots(figsize=(12, 6))

for row, (lbl, edges) in enumerate(zip(labels, edges_all)):
    y0 = row - 0.4
    ax2.broken_barh([(x_lo, edges[0] - x_lo)], (y0, 0.8), facecolors='steelblue', zorder=3)
    for k, eps in enumerate(eps_steps):
        ax2.broken_barh([(edges[k], edges[k+1] - edges[k])], (y0, 0.8),
                         facecolors=_grade_color(eps), zorder=3)
    ax2.broken_barh([(edges[-1], x_hi - edges[-1])], (y0, 0.8),
                     facecolors='white', edgecolors='#aaa', linewidth=0.3, zorder=3)
    ax2.text(x_hi + 0.002, row, f'  {lbl}', va='center', fontsize=9)

ax2.axvline(x_scatterer_1[0], color='green', lw=1.0, ls='--', alpha=0.8, zorder=5,
            label='baseline front centre')

# Max-shift arrow (Baseline -> 2λ)
y_arrow = len(labels) - 0.2
ax2.annotate('', xy=(x_scatterer_1[1], y_arrow), xytext=(x_scatterer_1[0], y_arrow),
             arrowprops=dict(arrowstyle='->', color='k', lw=0.9))
ax2.text((x_scatterer_1[0] + x_scatterer_1[1]) / 2, y_arrow + 0.15,
         f'max shift {(x_scatterer_1[1]-x_scatterer_1[0])*1e3:.1f} mm = 2λ',
         ha='center', va='bottom', fontsize=8)

ax2.set_xlim(x_lo, x_hi + 0.05)
ax2.set_ylim(-1, len(labels))
ax2.set_yticks(range(len(labels)))
ax2.set_yticklabels(labels)
ax2.invert_yaxis()
ax2.set_xlabel('x [m]', fontsize=11)
ax2.set_title(
    f'Graded wetting zone — all 8 scenarios  '
    f'(box width = {box_width*1e3:.1f} mm, total transition = {total_width*1e3:.1f} mm, '
    f'depth = {z_scatterer:.3f} m)',
    fontsize=11
)
ax2.legend(loc='upper right', fontsize=8)
ax2.grid(True, ls='--', alpha=0.3, axis='x')
plt.tight_layout()
plt.show()


# Generating Geometry Files and Merged B-scans
Only uncomment and run when necessary

In [ ]:
# input_baseline      = r'fluidflow_study/baseline/resolution_baseline.in'
# input_lambda_2      = r'fluidflow_study/shift_2lambda/resolution_2lambda.in'
# input_lambda_1      = r'fluidflow_study/shift_1lambda/resolution_1lambda.in'
# input_lambda_0p5    = r'fluidflow_study/shift_0p5lambda/resolution_0p5lambda.in'
# input_lambda_0p25   = r'fluidflow_study/shift_0p25lambda/resolution_0p25lambda.in'
# input_lambda_0p125  = r'fluidflow_study/shift_0p125lambda/resolution_0p125lambda.in'
# input_lambda_0p0625 = r'fluidflow_study/shift_0p0625lambda/resolution_0p0625lambda.in'
# input_lambda_0p03125 = r'fluidflow_study/shift_0p03125lambda/resolution_0p03125lambda.in'

# api(input_baseline, n=1, geometry_only=True)
# api(input_lambda_2, n=1, geometry_only=True)
# api(input_lambda_1, n=1, geometry_only=True)
# api(input_lambda_0p5, n=1, geometry_only=True)
# api(input_lambda_0p25, n=1, geometry_only=True)
# api(input_lambda_0p125, n=1, geometry_only=True)
# api(input_lambda_0p0625, n=1, geometry_only=True)
# api(input_lambda_0p03125, n=1, geometry_only=True)


In [ ]:
# filename_background    = r'fluidflow_study/background/background'
# filename_baseline      = r'fluidflow_study/baseline/resolution_baseline'
# filename_lambda_2      = r'fluidflow_study/shift_2lambda/resolution_2lambda'
# filename_lambda_1      = r'fluidflow_study/shift_1lambda/resolution_1lambda'
# filename_lambda_0p5    = r'fluidflow_study/shift_0p5lambda/resolution_0p5lambda'
# filename_lambda_0p25   = r'fluidflow_study/shift_0p25lambda/resolution_0p25lambda'
# filename_lambda_0p125  = r'fluidflow_study/shift_0p125lambda/resolution_0p125lambda'
# filename_lambda_0p0625 = r'fluidflow_study/shift_0p0625lambda/resolution_0p0625lambda'
# filename_lambda_0p03125 = r'fluidflow_study/shift_0p03125lambda/resolution_0p03125lambda'

# merge_files(filename_background, removefiles=True)
# merge_files(filename_baseline, removefiles=True)
# merge_files(filename_lambda_2, removefiles=True)
# merge_files(filename_lambda_1, removefiles=True)
# merge_files(filename_lambda_0p5, removefiles=True)
# merge_files(filename_lambda_0p25, removefiles=True)
# merge_files(filename_lambda_0p125, removefiles=True)
# merge_files(filename_lambda_0p0625, removefiles=True)
# merge_files(filename_lambda_0p03125, removefiles=True)


# Loading in the data and removing direct wave

In [ ]:
rxnumber    = 1
rxcomponent = 'Ez'

output_background, dt = get_output_data(
    'fluidflow_study/background/background_merged.out', rxnumber, rxcomponent
)

output_baseline, _ = get_output_data(
    'fluidflow_study/baseline/resolution_baseline_merged.out', rxnumber, rxcomponent
)

_paths = [
    'fluidflow_study/baseline/resolution_baseline_merged.out',
    'fluidflow_study/shift_2lambda/resolution_2lambda_merged.out',
    'fluidflow_study/shift_1lambda/resolution_1lambda_merged.out',
    'fluidflow_study/shift_0p5lambda/resolution_0p5lambda_merged.out',
    'fluidflow_study/shift_0p25lambda/resolution_0p25lambda_merged.out',
    'fluidflow_study/shift_0p125lambda/resolution_0p125lambda_merged.out',
    'fluidflow_study/shift_0p0625lambda/resolution_0p0625lambda_merged.out',
    'fluidflow_study/shift_0p03125lambda/resolution_0p03125lambda_merged.out',
]
raw_outputs      = [get_output_data(p, rxnumber, rxcomponent)[0] for p in _paths]
outputs_static   = [r - output_background for r in raw_outputs]      # background-subtracted

dt_ns   = dt * 1e9
n_t     = outputs_static[0].shape[0]
time_ns = np.arange(n_t) * dt_ns

# Convenience lists for the visualisation cells below
data_pre = [('Background', output_background)] + list(zip(labels, raw_outputs))
data_static = list(zip(labels, outputs_static))
labels_all = list(labels)           # all 8 labels (Baseline + 7 shifts)
labels = labels[1:]

print(f'dt = {dt_ns:.6f} ns,  n_t = {n_t},  t_max = {time_ns[-1]:.2f} ns')
print(f'Data shape (n_t × n_tr): {outputs_static[0].shape}  ({n_t} time samples × {n_traces} traces)')


# Importing Noise

In [ ]:
import pickle
from scipy import stats
with open('laplace_noise_model.pkl', 'rb') as f:
    noise_model = pickle.load(f)

with open('laplace_noise_model_pre_gain.pkl', 'rb') as f:
    noise_model_pre_gain = pickle.load(f)


# Visualisation

In [ ]:
extent_bscan = [x_traces[0], x_traces[-1], time_ns[-1], 0]

fig, axes = plt.subplots(1, 9, figsize=(25, 5), facecolor='w', sharey=True)
vmax = max(np.abs(d).max() for _, d in data_pre)

for i, (ax, (title, d)) in enumerate(zip(axes, data_pre)):
    im = ax.imshow(d, aspect='auto', cmap='seismic', extent=extent_bscan,
                   interpolation='nearest', vmin=-vmax, vmax=vmax)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('x [m]', fontsize=11)
    if ax is axes[0]:
        ax.set_ylabel('Time [ns]', fontsize=11)
    ax.grid(linestyle='-.', alpha=0.4)
    if i > 0:  # skip Background panel (no fluid front)
        ax.axvline(x_scatterer_1[i - 1], color='green', linestyle='--', linewidth=1)

fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02, label='Ez [V/m]')
fig.suptitle('GPR B-Scans — Background, Baseline and TimeLapsed Models', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(25, 5), facecolor='w', sharey=True)
vmax = max(np.abs(d).max() for _, d in data_static)

for i, (ax, (title, d)) in enumerate(zip(axes, data_static)):
    im = ax.imshow(d, aspect='auto', cmap='seismic', extent=extent_bscan,
                   interpolation='nearest', vmin=-vmax, vmax=vmax)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('x [m]', fontsize=11)
    if ax is axes[0]:
        ax.set_ylabel('Time [ns]', fontsize=11)
    ax.grid(linestyle='-.', alpha=0.4)
    ax.axvline(x_scatterer_1[i], color='green', linestyle='--', linewidth=1)

fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02, label='Ez [V/m]')
fig.suptitle('GPR B-Scans — Background Subtracted', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


Create Noisy Data

In [ ]:
# ── Apply the fitted Laplace noise model to every background-subtracted B-scan ──
# The Laplace fit (loc/scale) was estimated from the real, fully-processed data
# pipeline ('Spherical Gain' stage), whose amplitudes are ~4 orders of magnitude
# larger than these synthetic Ez B-scans. We keep the Laplace *shape* (loc=0,
# heavier tails than Gaussian) but rescale it so its std is 10% of each B-scan's
# own signal std — a light, realistic noise level rather than the raw fitted scale.
NOISE_LEVEL = 0.1   # target noise std as a fraction of each B-scan's signal std
rng = np.random.default_rng(0)

outputs_noisy = []
for raw in outputs_static:
    target_std    = NOISE_LEVEL * raw.std()
    scaled_scale  = target_std / np.sqrt(2)   # Var(Laplace) = 2 * scale**2
    synthetic_noise = stats.laplace.rvs(
        loc=noise_model_pre_gain['loc'], scale=scaled_scale,
        size=raw.shape, random_state=rng,
    )
    outputs_noisy.append(raw + synthetic_noise)

data_noisy = list(zip(labels_all, outputs_noisy))

print(f"Applied Laplace noise (shape from '{noise_model_pre_gain['stage']}' fit, "
      f"rescaled to {NOISE_LEVEL:.0%} of signal std) to {len(outputs_noisy)} B-scans")


In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(25, 5), facecolor='w', sharey=True)
vmax = max(np.abs(d).max() for _, d in data_noisy)

for i, (ax, (title, d)) in enumerate(zip(axes, data_noisy)):
    im = ax.imshow(d, aspect='auto', cmap='seismic', extent=extent_bscan,
                   interpolation='nearest', vmin=-vmax, vmax=vmax)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('x [m]', fontsize=11)
    if ax is axes[0]:
        ax.set_ylabel('Time [ns]', fontsize=11)
    ax.grid(linestyle='-.', alpha=0.4)
    ax.axvline(x_scatterer_1[i], color='green', linestyle='--', linewidth=1)

fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02, label='Ez [V/m]')
fig.suptitle('GPR B-Scans — With Synthetic Laplace Noise (10% of signal std)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# -- Frequency spectra: clean vs noisy B-scans -----------------------------------
# Row 1: amplitude spectrum (mean across traces) of each clean (background-
# subtracted) B-scan in data_static. Row 2: the same for the noisy B-scans in
# data_noisy -- compare rows to see how much broadband content the Laplace
# noise injection adds on top of the underlying Ricker wavelet bandwidth.
freqs_ghz = np.fft.rfftfreq(n_t, d=dt_ns)

fig, axes = plt.subplots(2, 8, figsize=(25, 8), sharex=True, sharey='row')
for ax, (label, d) in zip(axes[0], data_static):
    spec = np.abs(np.fft.rfft(d, axis=0)).mean(axis=1)
    ax.plot(freqs_ghz, spec, lw=0.9)
    ax.set_title(f'{label} (clean)', fontsize=11)
    ax.set_xlabel('Frequency [GHz]')
axes[0, 0].set_ylabel('|FFT| (mean over traces)')

for ax, (label, d) in zip(axes[1], data_noisy):
    spec = np.abs(np.fft.rfft(d, axis=0)).mean(axis=1)
    ax.plot(freqs_ghz, spec, lw=0.9, color='C1')
    ax.set_title(f'{label} (noisy)', fontsize=11)
    ax.set_xlabel('Frequency [GHz]')
axes[1, 0].set_ylabel('|FFT| (mean over traces)')

plt.suptitle(f'B-scan Frequency Spectra -- Clean vs Noisy  |  f_c={f_c_GHz} GHz',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


# Saving the Data

In [ ]:
# ── Save non-migrated (raw, background-subtracted) B-scan data ────────────────
# Stacks all 8 scenarios (Baseline + 7 shifts) from data_static into a single
# array and saves to static_results.npz.
# Index 0 = Baseline, indices 1-7 = shift scenarios matching labels_all[1:].

static_all = np.stack([d.astype(float) for _, d in data_static], axis=0)  # (8, n_t, n_traces)

save_path_static = STUDY_ROOT / "static_results.npz"

np.savez_compressed(
    save_path_static,
    data_static       = static_all,                                     # (8, n_t, n_traces)
    scenarios         = np.array(labels_all, dtype="U20"),               # (8,)  Baseline + 7 shifts
    separation_lambda = np.array([0, 2, 1, 0.5, 0.25, 0.125, 0.0625, 0.03125]),  # (8,)  0 = baseline
    x_traces          = x_traces,
    time_ns           = time_ns,
    dt                = np.float64(dt),
    x_scatterer       = np.array(x_scatterer_1),     # (8,) wetting-zone centre per scenario
    z_scatterer       = np.float64(z_scatterer),
    z_top             = np.float64(z_top),
    box_width         = np.float64(box_width),
    total_width       = np.float64(total_width),
)

print(f"Saved → {save_path_static}")
print(f"\nStacked array  (n_scenarios={static_all.shape[0]}, n_t={static_all.shape[1]}, n_tr={static_all.shape[2]}):")
print(f"  data_static   {static_all.shape}")
print(f"\nUsage example:")
print(f"  d = np.load(str(STUDY_ROOT / 'static_results.npz'), allow_pickle=False)")
print(f"  d['data_static'][0]   # Baseline B-scan → shape (n_t, n_tr)")
print(f"  d['data_static'][3]   # ½λ shift B-scan → shape (n_t, n_tr)")


# Kirchhoff Migration

Delay-and-sum migration via the **PyLops zero-offset Kirchhoff operator**.
Each dataset is preprocessed (tapering + t0 shift — see cell above) before being migrated.
Green stars mark the true wetting-zone-centre positions.


In [ ]:
import sys, pathlib
_here = pathlib.Path.cwd()
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))
from helper_functions.migration import (PylopsKirchoffMigration, gazdag_migration,
                                        write_backprop_files, dispersion_limited_cutoff,
                                        lowpass_filter_excitation)


## Tapering & t0 Shift

Two pre-processing steps applied to every B-scan before migration:

| Step | Purpose |
|------|---------|
| **Exponential decay taper** (`taper_decay_ns`) | Suppresses late arrivals, emphasises hyperbola apices |
| **Cosine end taper** (`taper_end_ns`) | Zeros the final portion of each trace to remove hyperbola tails |
| **t0 shift** | Removes the Ricker wavelet's built-in peak delay so that t = 0 aligns with the surface |

Adjust the parameters in the cell below and re-run to see their effect on the B-scan.


In [ ]:
# ── Taper & t0-shift parameters ───────────────────────────────────────────────
taper_end_ns   = 14.0   # cosine ramp onset [ns]: zeros hyperbola tails beyond this time
taper_decay_ns = 7.0    # exponential decay time constant [ns]: de-emphasises late arrivals
ANGLE_AP       = 40     # Kirchhoff aperture half-angle [degrees]

# ── Functions ─────────────────────────────────────────────────────────────────
def make_end_taper(n_samples, taper_samples):
    win = np.ones(n_samples)
    if taper_samples > 0:
        ramp = 0.5 * (1 + np.cos(np.pi * np.arange(taper_samples) / taper_samples))
        win[-taper_samples:] = ramp
    return win

def make_decay_taper(time_ns, tau_ns):
    return np.exp(-time_ns / tau_ns)

def preprocess(data_nt_ntr, dt_ns, t0_ns, time_ns):
    bscan   = data_nt_ntr.T.copy()                              # → (n_tr, n_t)
    n_t     = bscan.shape[1]
    w_end   = make_end_taper(n_t, int(round(taper_end_ns / dt_ns)))
    w_decay = make_decay_taper(time_ns, taper_decay_ns)
    tapered = bscan * (w_end * w_decay)[np.newaxis, :]
    t0_samp = int(round(t0_ns / dt_ns))
    shifted = np.roll(tapered, -t0_samp, axis=1)
    shifted[:, -t0_samp:] = 0.0
    return tapered, shifted

# ── Visualise effect on the 2λ dataset ────────────────────────────────────────
_demo = outputs_static[1]    # (n_t, n_tr) — 2λ background-subtracted
_raw_tr, _shifted = preprocess(_demo, dt_ns, t0_ns, time_ns)
_tapered = _raw_tr   # preprocess returns tapered as first output

w_end_vis   = make_end_taper(n_t, int(round(taper_end_ns   / dt_ns)))
w_decay_vis = make_decay_taper(time_ns, taper_decay_ns)
taper_combined = w_end_vis * w_decay_vis

mid_tr = n_traces // 2   # representative central trace

# --- Row 1: single-trace before/after ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(time_ns, _demo[:, mid_tr], 'b', lw=0.8, label='Raw')
axes[0].plot(time_ns, _tapered[mid_tr], 'r', lw=0.8, label='Tapered')
axes[0].axvline(taper_end_ns, color='gray', ls='--', lw=1.2, label=f'End onset {taper_end_ns} ns')
axes[0].set(xlabel='Time [ns]', ylabel='Ez [V/m]', title=f'Trace #{mid_tr} — raw vs tapered')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].plot(time_ns, _tapered[mid_tr], 'r', lw=0.8, label='Tapered')
axes[1].plot(time_ns, _shifted[mid_tr], 'g', lw=0.8, label=f't0-shifted (−{t0_ns:.3f} ns)')
axes[1].set(xlabel='Time [ns]', ylabel='Ez [V/m]', title=f'Trace #{mid_tr} — tapered vs t0-shifted')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

axes[2].plot(time_ns, taper_combined, 'k', lw=1.5)
axes[2].axvline(taper_end_ns, color='gray', ls='--', lw=1.2, label=f'End taper onset')
axes[2].set(xlabel='Time [ns]', ylabel='Weight [–]',
            title=f'Combined taper  (decay τ={taper_decay_ns} ns, end from {taper_end_ns} ns)',
            ylim=(0, 1.05))
axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)

plt.suptitle('Effect of Tapering and t0 Shift — 2λ dataset, single trace', fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

# --- Row 2: full B-scan comparison ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
kw = dict(aspect='auto', cmap='seismic', extent=extent_bscan, origin='upper')

vmax0 = np.max(np.abs(_demo)) * 0.5
axes[0].imshow(_demo, **kw, vmin=-vmax0, vmax=vmax0)
axes[0].set(title='Raw (background subtracted)', xlabel='x [m]', ylabel='Time [ns]')

vmax1 = np.max(np.abs(_tapered)) * 0.5
axes[1].imshow(_tapered.T, **kw, vmin=-vmax1, vmax=vmax1)
axes[1].set(title=f'Tapered  (decay τ={taper_decay_ns} ns, end from {taper_end_ns} ns)', xlabel='x [m]')

vmax2 = np.max(np.abs(_shifted)) * 0.5
axes[2].imshow(_shifted.T, **kw, vmin=-vmax2, vmax=vmax2)
axes[2].set(title=f't0-shifted (−{t0_ns:.3f} ns)', xlabel='x [m]')

plt.suptitle('B-scan effect of tapering and t0 shift — 2λ dataset', fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
# ── Kirchhoff migration on all 8 datasets ─────────────────────────────────────────
migrated = {}

for label, raw in zip(labels_all, outputs_static):
    _, shifted = preprocess(raw, dt_ns, t0_ns, time_ns)   # (n_tr, n_t), t0-shifted
    print(f'[{label}] Running Kirchhoff ...', end=' ', flush=True)
    t0 = _time.perf_counter()
    img = PylopsKirchoffMigration(
        shifted, time_ns, x_traces, v_ice, z_img,
        f0=f_c_GHz, angleaperture=ANGLE_AP
    )
    print(f'{_time.perf_counter()-t0:.1f} s  shape={img.shape}')
    migrated[label] = img


In [ ]:
# ── Plot: full extent ──────────────────────────────────────────────────────────────
extent_mig = [x_traces[0], x_traces[-1], z_img[-1], z_img[0]]

fig, axes = plt.subplots(2, 4, figsize=(24, 10), sharex=True, sharey=True)
for i, (ax, (label, img)) in enumerate(zip(axes.ravel(), migrated.items())):
    vmax = np.max(np.abs(img)) * 0.8
    ax.imshow(img, aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax,
              extent=extent_mig, origin='upper')
    ax.plot(x_scatterer_1[0], z_top, 'g*', ms=10, zorder=5, label='x_s1 baseline')
    ax.plot(x_scatterer_1[i], z_top, 'g^', ms=10, zorder=5, label='x_s1 current')
    ax.set_title(f'Kirchhoff — {label}', fontsize=11)
    ax.set_xlabel('x [m]'); ax.set_ylabel('Depth z [m]')

plt.suptitle(
    f'Kirchhoff Migration — All 8 Datasets  |  f_c={f_c_GHz} GHz  |  aperture={ANGLE_AP}°  |  z_top={z_top:.3f} m',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

# ── Plot: zoomed on scatterer region ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(24, 10), sharex=True, sharey=True)
for i, (ax, (label, img)) in enumerate(zip(axes.ravel(), migrated.items())):
    vmax = np.max(np.abs(img)) * 0.8
    ax.imshow(img, aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax,
              extent=extent_mig, origin='upper')
    ax.plot(x_scatterer_1[0], z_top, 'g*', ms=10, zorder=5, label='x_s1 baseline')
    ax.plot(x_scatterer_1[i], z_top, 'g^', ms=10, zorder=5, label='x_s1 current')
    ax.set_xlim(1.5, 2.5); ax.set_ylim(0.75, 0.55)
    ax.set_title(f'{label} (zoomed)', fontsize=11)
    ax.set_xlabel('x [m]'); ax.set_ylabel('Depth z [m]')
    ax.legend(fontsize=8, loc='upper right')

plt.suptitle(
    f'Kirchhoff Migration (zoomed)  |  f_c={f_c_GHz} GHz  |  aperture={ANGLE_AP}°',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()


In [ ]:
# ── Kirchhoff timelapse differences (migrated − migrated_baseline) ─────────────
migrated_diff = {label: migrated[label] - migrated['Baseline'] for label in labels}

fig, axes = plt.subplots(2, 4, figsize=(18, 10), sharex=True, sharey=True)
axes.ravel()[-1].set_visible(False)
for i, (ax, (label, diff)) in enumerate(zip(axes.ravel(), migrated_diff.items())):
    vmax = np.max(np.abs(diff)) * 0.8
    ax.imshow(diff, aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax,
              extent=extent_mig, origin='upper')
    ax.plot(x_scatterer_1[0],     z_top, 'g*', ms=10, zorder=5, label='x_s1 baseline')
    ax.plot(x_scatterer_1[i + 1], z_top, 'g^', ms=10, zorder=5, label='x_s1 timelapsed')
    ax.set_title(f'TimeLapse diff — {label}', fontsize=11)
    ax.set_xlabel('x [m]'); ax.set_ylabel('Depth z [m]')
    ax.legend(fontsize=8, loc='upper right')

plt.suptitle(
    f'Kirchhoff Migration — TimeLapse Differences (migrated − migrated_baseline)  |  f_c={f_c_GHz} GHz',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

# ── Zoomed ────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 10), sharex=True, sharey=True)
axes.ravel()[-1].set_visible(False)
for i, (ax, (label, diff)) in enumerate(zip(axes.ravel(), migrated_diff.items())):
    vmax = np.max(np.abs(diff)) * 0.8
    ax.imshow(diff, aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax,
              extent=extent_mig, origin='upper')
    ax.plot(x_scatterer_1[0],     z_top, 'g*', ms=10, zorder=5, label='x_s1 baseline')
    ax.plot(x_scatterer_1[i + 1], z_top, 'g^', ms=10, zorder=5, label='x_s1 timelapsed')
    ax.set_xlim(1.5, 2.5); ax.set_ylim(0.75, 0.55)
    ax.set_title(f'TimeLapse diff — {label} (zoomed)', fontsize=11)
    ax.set_xlabel('x [m]'); ax.set_ylabel('Depth z [m]')
    ax.legend(fontsize=8, loc='upper right')

plt.suptitle(
    f'Kirchhoff Migration — TimeLapse Differences (zoomed)',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()


# Gazdag Phase-Shift Migration

f-k domain migration via **PyLops `PhaseShift`** downward continuation.
Uses the same preprocessed (tapered + t0-shifted) B-scans as Kirchhoff, but operates in the
frequency-wavenumber domain: each depth step phase-shifts the wavefield by `kz·dz`.

Key implementation details:
- `v_mig = v_ice / 2` (exploding-reflector half-velocity) used internally
- 100 % spatial zero-padding each side to avoid wrap-around in kx
- Evanescent bins (`|kx| > |f|/v_mig`) are zeroed **before** the depth loop to prevent smile artefacts accumulating at the t = 0 imaging condition
- Green stars mark the true wetting-zone-centre positions


_____________
Add padding in space and time up to the next power of 2
_____________

In [ ]:
import pylops


## Run Migration on All Datasets

In [ ]:
# ── Gazdag phase-shift migration on all 8 datasets ────────────────────────────────
migrated_gz = {}

for label, raw in zip(labels_all, outputs_static):
    _, shifted = preprocess(raw, dt_ns, t0_ns, time_ns)   # (n_tr, n_t), t0-shifted
    print(f'[{label}] Running Gazdag ...', flush=True)
    t0 = _time.perf_counter()
    img = gazdag_migration(
        shifted.T,       # (n_t, n_tr) — time first
        x_traces, time_ns, z_img, v_ice
    )
    print(f'  Done in {_time.perf_counter()-t0:.1f} s  shape={img.shape}')
    migrated_gz[label] = img


In [ ]:
# ── Plot: full extent ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(24, 10), sharex=True, sharey=True)
for i, (ax, (label, img)) in enumerate(zip(axes.ravel(), migrated_gz.items())):
    vmax = np.max(np.abs(img)) * 0.8
    ax.imshow(img, aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax,
              extent=extent_mig, origin='upper')
    ax.plot(x_scatterer_1[0], z_top, 'g*', ms=10, zorder=5, label='x_s1 baseline')
    ax.plot(x_scatterer_1[i], z_top, 'g^', ms=10, zorder=5, label='x_s1 current')
    ax.set_title(f'Gazdag — {label}', fontsize=11)
    ax.set_xlabel('x [m]'); ax.set_ylabel('Depth z [m]')

plt.suptitle(
    f'Gazdag Phase-Shift Migration — All 8 Datasets  |  f_c={f_c_GHz} GHz  |  z_top={z_top:.3f} m',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

# ── Plot: zoomed on scatterer region ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(24, 10), sharex=True, sharey=True)
for i, (ax, (label, img)) in enumerate(zip(axes.ravel(), migrated_gz.items())):
    vmax = np.max(np.abs(img)) * 0.8
    ax.imshow(img, aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax,
              extent=extent_mig, origin='upper')
    ax.plot(x_scatterer_1[0], z_top, 'g*', ms=10, zorder=5, label='x_s1 baseline')
    ax.plot(x_scatterer_1[i], z_top, 'g^', ms=10, zorder=5, label='x_s1 current')
    ax.set_xlim(1.5, 2.5); ax.set_ylim(0.75, 0.55)
    ax.set_title(f'{label} (zoomed)', fontsize=11)
    ax.set_xlabel('x [m]'); ax.set_ylabel('Depth z [m]')
    ax.legend(fontsize=8, loc='upper right')

plt.suptitle(
    f'Gazdag Phase-Shift Migration (zoomed)  |  f_c={f_c_GHz} GHz',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()


In [ ]:
# ── Gazdag timelapse differences (migrated − migrated_baseline) ──────────────────────
migrated_gz_diff = {label: migrated_gz[label] - migrated_gz['Baseline'] for label in labels}

fig, axes = plt.subplots(3, 3, figsize=(18, 10), sharex=True, sharey=True)
axes.ravel()[-2].set_visible(False)
axes.ravel()[-1].set_visible(False)
for i, (ax, (label, diff)) in enumerate(zip(axes.ravel(), migrated_gz_diff.items())):
    vmax = np.max(np.abs(diff)) * 0.8
    ax.imshow(diff, aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax,
              extent=extent_mig, origin='upper')
    ax.plot(x_scatterer_1[0],     z_top, 'g*', ms=10, zorder=5, label='x_s1 baseline')
    ax.plot(x_scatterer_1[i + 1], z_top, 'g^', ms=10, zorder=5, label='x_s1 timelapsed')
    ax.set_title(f'TimeLapse diff — {label}', fontsize=11)
    ax.set_xlabel('x [m]'); ax.set_ylabel('Depth z [m]')
    ax.legend(fontsize=8, loc='upper right')

plt.suptitle(
    f'Gazdag Migration — TimeLapse Differences (migrated − migrated_baseline)  |  f_c={f_c_GHz} GHz',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

# ── Zoomed ────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(18, 10), sharex=True, sharey=True)
axes.ravel()[-2].set_visible(False)
axes.ravel()[-1].set_visible(False)
for i, (ax, (label, diff)) in enumerate(zip(axes.ravel(), migrated_gz_diff.items())):
    vmax = np.max(np.abs(diff)) * 0.8
    ax.imshow(diff, aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax,
              extent=extent_mig, origin='upper')
    ax.plot(x_scatterer_1[0],     z_top, 'g*', ms=10, zorder=5, label='x_s1 baseline')
    ax.plot(x_scatterer_1[i + 1], z_top, 'g^', ms=10, zorder=5, label='x_s1 timelapsed')
    ax.set_xlim(1.5, 2.5); ax.set_ylim(0.75, 0.55)
    ax.set_title(f'TimeLapse diff — {label} (zoomed)', fontsize=11)
    ax.set_xlabel('x [m]'); ax.set_ylabel('Depth z [m]')
    ax.legend(fontsize=8, loc='upper right')

plt.suptitle(
    f'Gazdag Migration — TimeLapse Differences (zoomed)',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()


# Back-Propagation Migration

Time-reversal migration: re-inject the time-reversed, normalised B-scan traces at the
original receiver positions through a **half-velocity medium** (eps_r × 4 so v → v_ice/2).

Under the exploding-reflector hypothesis every scatterer focuses simultaneously at

> **t_focus = T − t0_ns**

where T is the total simulation window. Snapshots are captured in a narrow window just
before t_focus so the collapsed wavefield image can be extracted.

| Parameter | Value |
|-----------|-------|
| Injection positions | Rx locations (Tx + 0.1 m offset) |
| Medium | eps_r × 4 → v = v_ice / 2 |
| Source stride | every `STRIDE`-th trace (tunes performance vs coverage) |


In [ ]:
# ── Back-propagation parameters ──────────────────────────────────────────────
# Sources placed at Tx-Rx midpoints (x_traces) — consistent with exploding-
# reflector half-velocity (v_mig = v/2) used in gazdag_migration.
slugs_all = ['baseline', '2lambda', '1lambda', '0p5lambda', '0p25lambda', '0p125lambda', '0p0625lambda', '0p03125lambda']
STRIDE   = 1    # use every Nth trace — reduces gprMax source count
N_SNAP   = 30   # number of snapshots in the focus window
SNAP_WIN = 1.0  # [ns] window before focus time to capture

# ── Generate files for all 8 datasets ────────────────────────────────────────
print(f'Back-propagation file generation  (stride={STRIDE}, {N_SNAP} snapshots)\n')
bp_paths = {}
for label, slug, raw in zip(labels_all, slugs_all, outputs_static):
    tapered, _ = preprocess(raw, dt_ns, t0_ns, time_ns)
    in_path, n_src, n_snaps, t_focus = write_backprop_files(
        STUDY_ROOT, label, slug, tapered, dt_ns, x_traces,
        t0_ns, eps_r_ice, v_ice, stride=STRIDE, n_snap=N_SNAP, snap_win=SNAP_WIN
    )
    bp_paths[label] = in_path
    exc_mb = (in_path.parent / 'excitation.txt').stat().st_size / 1e6
    print(f'  [{label}]  {n_src} sources  |  {n_snaps} snapshots  |  '
          f'focus={t_focus:.2f} ns  |  excitation={exc_mb:.1f} MB')
    print(f'           {in_path}')


In [ ]:
import pyvista

# ── Load focus frame for all 8 datasets ────────────────────────────────────────────
T_ns_bp    = n_t * dt_ns
t_focus_ns = T_ns_bp - t0_ns
t_start_ns = max(0.0, t_focus_ns - SNAP_WIN)
dt_s       = dt_ns * 1e-9
snap_step_ref = max(1, int((T_ns_bp*1e-9 - t_start_ns*1e-9) / (max(1, N_SNAP - 1) * dt_s)))

focus_frames = {}
for i, (label, slug) in enumerate(zip(labels_all, slugs_all)):
    snap_dir   = STUDY_ROOT / 'backprop' / slug / f'backprop_{slug}_snaps'
    snap_files = sorted(
        snap_dir.glob('bp_snap*.vti'),
        key=lambda p: int(p.stem.replace('bp_snap', ''))
    )
    if not snap_files:
        print(f'[{label}] No snapshots in {snap_dir.name} — skipping')
        continue

    snap_times_ns = t_start_ns + np.arange(len(snap_files)) * snap_step_ref * dt_ns
    idx_focus     = min(int(np.argmin(np.abs(snap_times_ns - t_focus_ns))),
                        len(snap_files) - 1)

    snaps_mag, snaps_ez = [], []
    for p in snap_files:
        mesh   = pyvista.read(str(p))
        e_data = np.array(mesh['E-field'])
        snaps_mag.append(np.linalg.norm(e_data, axis=1).reshape(1000, 4000))
        snaps_ez.append(e_data[:, 2].reshape(1000, 4000))

    focus_frames[label] = {
        'mag':      np.stack(snaps_mag)[idx_focus],
        'ez':       np.stack(snaps_ez)[idx_focus],
        't_actual': snap_times_ns[idx_focus],
        'i':        i,
    }
    print(f'[{label}]  {len(snap_files)} snaps  |  focus idx={idx_focus}'
          f'  t={focus_frames[label]["t_actual"]:.3f} ns')

extent_full = [0, 4.0, 0, 1]
margin_x    = 0.4
margin_y    = 0.12

# Zoom x-window fixed to cover the widest (2λ) shift in all shared-axis panels
x_zoom_lo = x_scatterer_1[0] - margin_x
x_zoom_hi = x_scatterer_1[1] + margin_x   # x_scatterer_1[1] = 2λ position

def _add_markers(ax, i):
    ax.plot(x_scatterer_1[0], y_scatterer, 'g*', ms=10, zorder=5, label='x_s1 baseline')
    ax.plot(x_scatterer_1[i], y_scatterer, 'g^', ms=10, zorder=5, label='x_s1 current')

# ── |E| magnitude — full extent 2×4 grid ───────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(24, 10), sharex=True, sharey=True)
for ax, (label, frame) in zip(axes.ravel(), focus_frames.items()):
    ax.imshow(frame['mag'], aspect='auto', cmap='inferno',
              extent=extent_full, origin='lower')
    _add_markers(ax, frame['i'])
    ax.axhline(y_surface, color='cyan', lw=0.8, ls='--')
    ax.set_title(f'{label}  |  t={frame["t_actual"]:.2f} ns', fontsize=11)
    ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
axes.ravel()[0].legend(fontsize=8, loc='upper right')
plt.suptitle(f'Back-Propagation |E| — All 8 Datasets  |  focus at {t_focus_ns:.2f} ns',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# ── |E| magnitude — zoomed 2×4 grid ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(24, 10), sharex=True, sharey=True)
for ax, (label, frame) in zip(axes.ravel(), focus_frames.items()):
    ax.imshow(frame['mag'], aspect='auto', cmap='inferno',
              extent=extent_full, origin='lower')
    _add_markers(ax, frame['i'])
    ax.set_xlim(x_zoom_lo, x_zoom_hi)
    ax.set_ylim(y_scatterer - margin_y, y_scatterer + margin_y)
    ax.set_title(f'{label} (zoomed)', fontsize=11)
    ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
    ax.legend(fontsize=8, loc='upper right')
plt.suptitle(f'Back-Propagation |E| (zoomed)  |  focus at {t_focus_ns:.2f} ns',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# ── Ez component — full extent 2×4 grid ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(24, 10), sharex=True, sharey=True)
for ax, (label, frame) in zip(axes.ravel(), focus_frames.items()):
    ax.imshow(frame['ez'], aspect='auto', cmap='seismic',
              extent=extent_full, origin='lower')
    _add_markers(ax, frame['i'])
    ax.axhline(y_surface, color='cyan', lw=0.8, ls='--')
    ax.set_title(f'{label}  |  t={frame["t_actual"]:.2f} ns', fontsize=11)
    ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
axes.ravel()[0].legend(fontsize=8, loc='upper right')
plt.suptitle(f'Back-Propagation Ez — All 8 Datasets  |  focus at {t_focus_ns:.2f} ns',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# ── Ez component — zoomed 2×4 grid ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(24, 10), sharex=True, sharey=True)
for ax, (label, frame) in zip(axes.ravel(), focus_frames.items()):
    ax.imshow(frame['ez'], aspect='auto', cmap='seismic',
              extent=extent_full, origin='lower')
    _add_markers(ax, frame['i'])
    ax.set_xlim(x_zoom_lo, x_zoom_hi)
    ax.set_ylim(y_scatterer - margin_y, y_scatterer + margin_y)
    ax.set_title(f'{label} (zoomed)', fontsize=11)
    ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
    ax.legend(fontsize=8, loc='upper right')
plt.suptitle(f'Back-Propagation Ez (zoomed)  |  focus at {t_focus_ns:.2f} ns',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── Back-propagation timelapse differences (Ez − Ez_baseline) ─────────────────────
bp_ez_diff = {label: focus_frames[label]['ez'] - focus_frames['Baseline']['ez']
              for label in labels}

fig, axes = plt.subplots(2, 4, figsize=(18, 10), sharex=True, sharey=True)
axes.ravel()[-1].set_visible(False)
for i, (ax, (label, diff)) in enumerate(zip(axes.ravel(), bp_ez_diff.items())):
    vmax = np.percentile(np.abs(diff), 100)
    ax.imshow(diff, aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax,
              extent=extent_full, origin='lower')
    ax.plot(x_scatterer_1[0],     y_scatterer, 'g*', ms=10, zorder=5, label='x_s1 baseline')
    ax.plot(x_scatterer_1[i + 1], y_scatterer, 'g^', ms=10, zorder=5, label='x_s1 timelapsed')
    ax.set_title(f'TimeLapse diff Ez — {label}', fontsize=11)
    ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
    ax.legend(fontsize=8, loc='upper right')
plt.suptitle(
    f'Back-Propagation — TimeLapse Differences Ez (Ez − Ez_baseline)  |  focus at {t_focus_ns:.2f} ns',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

# ── Zoomed ────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 10), sharex=True, sharey=True)
axes.ravel()[-1].set_visible(False)
for i, (ax, (label, diff)) in enumerate(zip(axes.ravel(), bp_ez_diff.items())):
    vmax = np.percentile(np.abs(diff), 100)
    ax.imshow(diff, aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax,
              extent=extent_full, origin='lower')
    ax.plot(x_scatterer_1[0],     y_scatterer, 'g*', ms=10, zorder=5, label='x_s1 baseline')
    ax.plot(x_scatterer_1[i + 1], y_scatterer, 'g^', ms=10, zorder=5, label='x_s1 timelapsed')
    ax.set_xlim(x_zoom_lo, x_zoom_hi)
    ax.set_ylim(y_scatterer - margin_y, y_scatterer + margin_y)
    ax.set_title(f'TimeLapse diff Ez — {label} (zoomed)', fontsize=11)
    ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
    ax.legend(fontsize=8, loc='upper right')
plt.suptitle(
    f'Back-Propagation — TimeLapse Differences Ez (zoomed)',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()


# Analysis

In [ ]:
from scipy.signal import hilbert as scipy_hilbert
from scipy.interpolate import RegularGridInterpolator

# ── Collect timelapse difference images: rows = separations, cols = methods ─
methods = ['Kirchhoff', 'Gazdag', 'Back-prop']
n_m, n_s = len(methods), len(labels)
imgs = [[None] * n_m for _ in range(n_s)]

for i, lbl in enumerate(labels):
    # Kirchhoff: use precomputed diff dict; fall back to inline if stale kernel
    try:
        imgs[i][0] = migrated_diff[lbl]
    except (KeyError, NameError):
        if 'Baseline' in migrated and lbl in migrated:
            imgs[i][0] = migrated[lbl] - migrated['Baseline']
    # Gazdag: same pattern
    try:
        imgs[i][1] = migrated_gz_diff[lbl]
    except (KeyError, NameError):
        if 'Baseline' in migrated_gz and lbl in migrated_gz:
            imgs[i][1] = migrated_gz[lbl] - migrated_gz['Baseline']

# Back-propagation — reproject bp_ez_diff (already in memory) to migration grid
x_ax_bp = np.linspace(0, 4.0, 4000)
y_ax_bp = np.linspace(0, 1.0, 1000)
y_mig   = y_surface - z_img
Yq, Xq  = np.meshgrid(y_mig, x_traces, indexing='ij')

for i, lbl in enumerate(labels):
    if lbl not in bp_ez_diff:
        print(f'[{lbl}] bp_ez_diff missing — skipping')
        continue
    interp = RegularGridInterpolator(
        (y_ax_bp, x_ax_bp), bp_ez_diff[lbl],
        method='linear', bounds_error=False, fill_value=0.0
    )
    imgs[i][2] = interp((Yq, Xq))
    print(f'[{lbl}] back-prop diff reprojected  shape={imgs[i][2].shape}')

# ── Scatterer depth coordinates ──────────────────────────────────────────────
# Kirchhoff/Gazdag: z_top = depth to wetting-zone-centre top [m]
# Back-prop: gprMax y-axis runs 0–1 m from bottom; convert to depth from surface
bp_z_top = 0.9 - y_scatterer   # = z_scatterer ≈ 0.676 m

# ── Shared plot helper ───────────────────────────────────────────────────────
extent_mig = [x_traces[0], x_traces[-1], z_img[-1], z_img[0]]
xlim = (1.3, 2.7)
ylim = (0.80, 0.54)

def _comparison_figure(envelope=False):
    fig, axes = plt.subplots(n_s, n_m, figsize=(4.5*n_m, 3.0*n_s),
                             sharex=True, sharey=True)
    for i, lbl in enumerate(labels):
        avail = [imgs[i][j] for j in range(n_m) if imgs[i][j] is not None]
        if avail:
            row_vmax = max(
                np.percentile(np.abs(np.abs(scipy_hilbert(a, axis=0)) if envelope else a), 100)
                for a in avail
            )
        else:
            row_vmax = 1.0

        for j, method in enumerate(methods):
            ax  = axes[i, j]
            img = imgs[i][j]

            if i == 0:
                ax.set_title(method, fontsize=11, fontweight='bold', pad=4)
            if j == 0:
                ax.set_ylabel(lbl, fontsize=11, fontweight='bold',
                              rotation=0, labelpad=36, va='center')
            if i == n_s - 1:
                ax.set_xlabel('x [m]', fontsize=8)

            if img is None:
                ax.text(0.5, 0.5, 'N/A', transform=ax.transAxes,
                        ha='center', va='center', fontsize=12, color='#999')
                ax.set_facecolor('#f5f5f5')
                continue

            if envelope:
                disp  = np.abs(scipy_hilbert(img, axis=0))
                cmap, vmin, vmax = 'hot', 0, row_vmax
            else:
                disp  = img
                cmap, vmin, vmax = 'seismic', -row_vmax, row_vmax

            marker_z = bp_z_top if j == 2 else z_top

            ax.imshow(disp, aspect='auto', cmap=cmap,
                      extent=extent_mig, origin='upper')
            ax.plot(x_scatterer_1[0],     marker_z, 'g*', ms=8, zorder=5,
                    label='baseline')
            ax.plot(x_scatterer_1[i + 1], marker_z, 'g^', ms=8, zorder=5,
                    label='timelapsed')
            ax.axhline(marker_z, color='green', ls='--', lw=0.8, alpha=0.7)
            ax.set_xlim(*xlim)
            ax.set_ylim(*ylim)
            ax.tick_params(labelsize=7)

    mode = 'Hilbert Envelope' if envelope else 'Signed Amplitude'
    plt.suptitle(
        f'TimeLapse Migration Comparison — {mode}  |  f_c={f_c_GHz} GHz  '
        f'|  aperture={ANGLE_AP}°  |  ★ = baseline,  ▲ = timelapsed',
        fontsize=13, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    return fig

# ── Figure 1: signed amplitude ───────────────────────────────────────────────
fig1 = _comparison_figure(envelope=False)
plt.show()


In [ ]:
# ── Normalised lateral PSF of timelapse difference at scatterer depth ────────
# Rows = scenario (2λ … ¹⁄₁₆λ), Columns = method
# Each panel: horizontal slice of the timelapse difference image at z = marker_z,
# normalised to its absolute peak, showing how well each method localises the change.
# Blue  = signed amplitude (normalised to peak)
# Red   = Hilbert envelope
# Green dashed = timelapsed scatterer x-position
# Green dotted  = baseline scatterer x-position

x_win = 3.0 * wavelength_ice   # half-window around scatterer x-position [m]

fig_psf, axes_psf = plt.subplots(n_s, n_m,
                                  figsize=(4.0 * n_m, 2.6 * n_s),
                                  sharey=True)

for i, lbl in enumerate(labels):
    for j, method in enumerate(methods):
        ax  = axes_psf[i, j]
        img = imgs[i][j]

        if i == 0:
            ax.set_title(method, fontsize=10, fontweight='bold', pad=3)
        if j == 0:
            ax.set_ylabel(lbl, fontsize=9, fontweight='bold',
                          rotation=0, labelpad=34, va='center')
        if i == n_s - 1:
            ax.set_xlabel('x [m]', fontsize=8)

        if img is None:
            ax.text(0.5, 0.5, 'N/A', transform=ax.transAxes,
                    ha='center', va='center', fontsize=11, color='#999')
            ax.set_facecolor('#f4f4f4')
            continue

        # Depth index closest to the scatterer for this method
        marker_z = bp_z_top if j == 2 else z_top
        iz = int(np.argmin(np.abs(z_img - marker_z)))

        # Lateral profile at scatterer depth, normalised to absolute peak
        profile  = img[iz, :].copy()
        peak     = np.max(np.abs(profile))
        if peak > 0:
            profile /= peak
        envelope = np.abs(scipy_hilbert(profile))

        ax.plot(x_traces, profile,  color='steelblue', lw=0.9, label='Amplitude')
        ax.plot(x_traces, envelope, color='tomato',    lw=1.3, label='Envelope')
        ax.axvline(x_scatterer_1[i + 1], color='green', ls='--', lw=1.2,
                   label='x_s1 timelapsed')
        ax.axvline(x_scatterer_1[0],     color='green', ls=':',  lw=1.2,
                   label='x_s1 baseline')
        ax.axhline(0, color='k', lw=0.4, alpha=0.4)
        ax.set_xlim(x_scatterer_1[i + 1] - x_win, x_scatterer_1[i + 1] + x_win)
        ax.set_ylim(-1.15, 1.4)
        ax.tick_params(labelsize=7)
        ax.grid(alpha=0.2)

axes_psf[0, 0].legend(fontsize=7, loc='upper right')

plt.suptitle(
    'Normalised Lateral PSF — TimeLapse Difference at True Scatterer Depth\n'
    'blue = amplitude  |  red = Hilbert envelope  '
    '|  green dashed = x_s1 timelapsed  |  green dotted = x_s1 baseline',
    fontsize=11, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()


# Save Migrated Results as np arrays dictionary

In [ ]:
# ── Build structured numpy representation of all migrated difference results ──────
# imgs[scenario_idx][method_idx] is populated by the analysis cell above.
#   scenarios = labels   →  ["2λ", "1λ", "½λ", "¼λ", "⅛λ", "¹⁄₁₆λ", "¹⁄₃₂λ"]
#   methods              →  ["Kirchhoff", "Gazdag", "Back-prop"]
#   each image: shape (n_z, n_x) — timelapse difference (monitor − baseline)

_ref_shape = imgs[0][0].shape   # (n_z, n_x) reference from Kirchhoff 2λ

def _safe_stack(method_idx):
    planes = []
    for i in range(n_s):
        arr = imgs[i][method_idx]
        planes.append(arr.astype(float) if arr is not None
                      else np.full(_ref_shape, np.nan))
    return np.stack(planes, axis=0)

diff_kirchhoff = _safe_stack(0)   # (7, n_z, n_x)  timelapse differences
diff_gazdag    = _safe_stack(1)
diff_backprop  = _safe_stack(2)

# ── Save to disk ──────────────────────────────────────────────────────────────
save_path = STUDY_ROOT / "difference_migrated_results.npz"

payload = dict(
    kirchhoff_diff    = diff_kirchhoff,                                 # (7, n_z, n_x)
    gazdag_diff       = diff_gazdag,
    backprop_diff     = diff_backprop,
    scenarios         = np.array(labels,  dtype="U20"),                 # (7,)  string
    methods           = np.array(methods, dtype="U20"),                 # (3,)  string
    separation_lambda = np.array([2, 1, 0.5, 0.25, 0.125, 0.0625, 0.03125]),    # (7,)  float
    x_traces          = x_traces,                                       # (n_x,) m
    z_img             = z_img,                                          # (n_z,) m
    x_scatterer       = np.array(x_scatterer_1),                        # (8,) m
    z_scatterer       = np.float64(z_scatterer),
    z_top             = np.float64(z_top),
)

np.savez_compressed(save_path, **payload)

# ── In-memory convenience dict ────────────────────────────────────────────────
mig_diff = {
    "Kirchhoff": {lbl: diff_kirchhoff[i] for i, lbl in enumerate(labels)},
    "Gazdag":    {lbl: diff_gazdag[i]    for i, lbl in enumerate(labels)},
    "Back-prop": {lbl: diff_backprop[i]  for i, lbl in enumerate(labels)},
}

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"Saved → {save_path}")
print(f"\nDifference arrays  (n_scenarios={n_s}, n_z={_ref_shape[0]}, n_x={_ref_shape[1]}):")
for name, arr in [("kirchhoff_diff", diff_kirchhoff),
                  ("gazdag_diff",    diff_gazdag),
                  ("backprop_diff",  diff_backprop)]:
    absent = int(np.isnan(arr).all(axis=(1, 2)).sum())
    note   = f"  ({absent} scenario(s) absent → NaN)" if absent else ""
    print(f"  {name:<18}  {arr.shape}{note}")

print(f"\nAvailability  ✓ = computed   — = not run")
col_w = 14
print("  " + f"{'':>8}  " + "".join(f"{m:>{col_w}}" for m in methods))
for i, lbl in enumerate(labels):
    row = "  " + f"{lbl:>8}  "
    row += "".join(f"{'✓':>{col_w}}" if imgs[i][j] is not None else f"{'—':>{col_w}}"
                   for j in range(n_m))
    print(row)


In [ ]:
# ── Save normal (non-difference) migrated results from all methods ────────────
# Stacks all 8 scenarios (Baseline + 7 shifts) from migrated, migrated_gz,
# and focus_frames (back-prop), saving to migrated_results.npz.
# Index 0 = Baseline, indices 1-7 = shift scenarios matching labels_all[1:].

# ── Kirchhoff & Gazdag: direct stack over labels_all ─────────────────────────
kirchhoff_all = np.stack([migrated[lbl].astype(float)    for lbl in labels_all], axis=0)  # (8, n_z, n_x)
gazdag_all    = np.stack([migrated_gz[lbl].astype(float) for lbl in labels_all], axis=0)

# ── Back-prop: reproject focus_frames ez onto migration grid ──────────────────
x_ax_bp = np.linspace(0, 4.0, 4000)
y_ax_bp = np.linspace(0, 1.0, 1000)
y_mig   = y_surface - z_img
Yq, Xq  = np.meshgrid(y_mig, x_traces, indexing='ij')

from scipy.interpolate import RegularGridInterpolator as _RGI
backprop_planes = []
for lbl in labels_all:
    if lbl in focus_frames:
        interp = _RGI((y_ax_bp, x_ax_bp), focus_frames[lbl]['ez'],
                      method='linear', bounds_error=False, fill_value=0.0)
        backprop_planes.append(interp((Yq, Xq)).astype(float))
    else:
        backprop_planes.append(np.full((len(z_img), len(x_traces)), np.nan))
        print(f'[{lbl}] focus_frames missing — filled with NaN')
backprop_all = np.stack(backprop_planes, axis=0)  # (8, n_z, n_x)

# ── Save to disk ──────────────────────────────────────────────────────────────
save_path_mig = STUDY_ROOT / "migrated_results.npz"

np.savez_compressed(
    save_path_mig,
    kirchhoff         = kirchhoff_all,                                   # (8, n_z, n_x)
    gazdag            = gazdag_all,
    backprop          = backprop_all,
    scenarios         = np.array(labels_all, dtype="U20"),               # (8,)  Baseline + 7 shifts
    separation_lambda = np.array([0, 2, 1, 0.5, 0.25, 0.125, 0.0625, 0.03125]),  # (8,)  0 = baseline
    x_traces          = x_traces,
    z_img             = z_img,
    x_scatterer       = np.array(x_scatterer_1),
    z_scatterer       = np.float64(z_scatterer),
    z_top             = np.float64(z_top),
)

print(f"Saved → {save_path_mig}")
print(f"\nStacked arrays  (n_scenarios=8, n_z={kirchhoff_all.shape[1]}, n_x={kirchhoff_all.shape[2]}):")
for name, arr in [("kirchhoff",  kirchhoff_all),
                  ("gazdag",     gazdag_all),
                  ("backprop",   backprop_all)]:
    absent = int(np.isnan(arr).all(axis=(1, 2)).sum())
    note   = f"  ({absent} scenario(s) absent → NaN)" if absent else ""
    print(f"  {name:<12}  {arr.shape}{note}")
print(f"\nUsage examples:")
print(f"  d = np.load(str(STUDY_ROOT / 'migrated_results.npz'), allow_pickle=False)")
print(f"  d['kirchhoff'][0]   # Kirchhoff Baseline → shape (n_z, n_x)")
print(f"  d['gazdag'][3]      # Gazdag ½λ          → shape (n_z, n_x)")


# Application to Noisy Data

## Save Unmigrated Noisy B-scans

In [ ]:
# -- Save non-migrated (noisy, background-subtracted) B-scan data --------------
# Mirrors the "Saving the Data" step above, but for the Laplace-noise-augmented
# B-scans (data_noisy) created in the "Create Noisy Data" section.

noisy_static_all = np.stack([d.astype(float) for _, d in data_noisy], axis=0)  # (8, n_t, n_traces)

save_path_noisy_static = STUDY_ROOT / "static_results_noisy.npz"

np.savez_compressed(
    save_path_noisy_static,
    data_static       = noisy_static_all,                                    # (8, n_t, n_traces)
    scenarios         = np.array(labels_all, dtype="U20"),                   # (8,)  Baseline + 7 shifts
    separation_lambda = np.array([0, 2, 1, 0.5, 0.25, 0.125, 0.0625, 0.03125]),  # (8,)  0 = baseline
    x_traces          = x_traces,
    time_ns           = time_ns,
    dt                = np.float64(dt),
    x_scatterer       = np.array(x_scatterer_1),
    z_scatterer       = np.float64(z_scatterer),
    z_top             = np.float64(z_top),
    noise_level       = np.float64(NOISE_LEVEL),
)

print(f"Saved -> {save_path_noisy_static}")
print(f"\nStacked array  (n_scenarios={noisy_static_all.shape[0]}, n_t={noisy_static_all.shape[1]}, n_tr={noisy_static_all.shape[2]}):")
print(f"  data_static   {noisy_static_all.shape}")


## Kirchhoff Migration (Noisy)

In [ ]:
# -- Kirchhoff migration on all 8 noisy datasets --------------------------------
migrated_noisy = {}

for label, raw in zip(labels_all, outputs_noisy):
    _, shifted = preprocess(raw, dt_ns, t0_ns, time_ns)   # (n_tr, n_t), t0-shifted
    print(f'[{label}] Running Kirchhoff (noisy) ...', end=' ', flush=True)
    t0 = _time.perf_counter()
    img = PylopsKirchoffMigration(
        shifted, time_ns, x_traces, v_ice, z_img,
        f0=f_c_GHz, angleaperture=ANGLE_AP
    )
    print(f'{_time.perf_counter()-t0:.1f} s  shape={img.shape}')
    migrated_noisy[label] = img


In [ ]:
# -- Plot: full extent ----------------------------------------------------------
fig, axes = plt.subplots(2, 4, figsize=(24, 10), sharex=True, sharey=True)
for i, (ax, (label, img)) in enumerate(zip(axes.ravel(), migrated_noisy.items())):
    vmax = np.max(np.abs(img)) * 0.8
    im = ax.imshow(img, aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax,
              extent=extent_mig, origin='upper')
    ax.plot(x_scatterer_1[0], z_top, 'g*', ms=10, zorder=5, label='x_s1 baseline')
    ax.plot(x_scatterer_1[i], z_top, 'g^', ms=10, zorder=5, label='x_s1 current')
    ax.set_title(f'Kirchhoff (noisy) - {label}', fontsize=11)
    ax.set_xlabel('x [m]'); ax.set_ylabel('Depth z [m]')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle(
    f'Kirchhoff Migration - All 8 Noisy Datasets  |  f_c={f_c_GHz} GHz  |  aperture={ANGLE_AP} deg  |  z_top={z_top:.3f} m',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

# -- Plot: zoomed on scatterer region --------------------------------------------
fig, axes = plt.subplots(2, 4, figsize=(24, 10), sharex=True, sharey=True)
for i, (ax, (label, img)) in enumerate(zip(axes.ravel(), migrated_noisy.items())):
    vmax = np.max(np.abs(img)) * 0.8
    im = ax.imshow(img, aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax,
              extent=extent_mig, origin='upper')
    ax.plot(x_scatterer_1[0], z_top, 'g*', ms=10, zorder=5, label='x_s1 baseline')
    ax.plot(x_scatterer_1[i], z_top, 'g^', ms=10, zorder=5, label='x_s1 current')
    ax.set_xlim(1.5, 2.5); ax.set_ylim(0.75, 0.55)
    ax.set_title(f'{label} (noisy, zoomed)', fontsize=11)
    ax.set_xlabel('x [m]'); ax.set_ylabel('Depth z [m]')
    ax.legend(fontsize=8, loc='upper right')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle(
    f'Kirchhoff Migration - Noisy (zoomed)  |  f_c={f_c_GHz} GHz  |  aperture={ANGLE_AP} deg',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()


In [ ]:
# -- Kirchhoff timelapse differences (noisy migrated - noisy migrated_baseline) --
migrated_diff_noisy = {label: migrated_noisy[label] - migrated_noisy['Baseline'] for label in labels}

fig, axes = plt.subplots(2, 4, figsize=(18, 10), sharex=True, sharey=True)
axes.ravel()[-1].set_visible(False)
for i, (ax, (label, diff)) in enumerate(zip(axes.ravel(), migrated_diff_noisy.items())):
    vmax = np.max(np.abs(diff)) * 0.8
    im = ax.imshow(diff, aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax,
              extent=extent_mig, origin='upper')
    ax.plot(x_scatterer_1[0],     z_top, 'g*', ms=10, zorder=5, label='x_s1 baseline')
    ax.plot(x_scatterer_1[i + 1], z_top, 'g^', ms=10, zorder=5, label='x_s1 timelapsed')
    ax.set_title(f'TimeLapse diff (noisy) - {label}', fontsize=11)
    ax.set_xlabel('x [m]'); ax.set_ylabel('Depth z [m]')
    ax.legend(fontsize=8, loc='upper right')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle(
    f'Kirchhoff Migration - Noisy TimeLapse Differences (migrated - migrated_baseline)  |  f_c={f_c_GHz} GHz',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

# -- Zoomed -----------------------------------------------------------------------
fig, axes = plt.subplots(2, 4, figsize=(18, 10), sharex=True, sharey=True)
axes.ravel()[-1].set_visible(False)
for i, (ax, (label, diff)) in enumerate(zip(axes.ravel(), migrated_diff_noisy.items())):
    vmax = np.max(np.abs(diff)) * 0.8
    im = ax.imshow(diff, aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax,
              extent=extent_mig, origin='upper')
    ax.plot(x_scatterer_1[0],     z_top, 'g*', ms=10, zorder=5, label='x_s1 baseline')
    ax.plot(x_scatterer_1[i + 1], z_top, 'g^', ms=10, zorder=5, label='x_s1 timelapsed')
    ax.set_xlim(1.5, 2.5); ax.set_ylim(0.75, 0.55)
    ax.set_title(f'TimeLapse diff (noisy) - {label} (zoomed)', fontsize=11)
    ax.set_xlabel('x [m]'); ax.set_ylabel('Depth z [m]')
    ax.legend(fontsize=8, loc='upper right')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle(
    f'Kirchhoff Migration - Noisy TimeLapse Differences (zoomed)',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()


## Gazdag Phase-Shift Migration (Noisy)

In [ ]:
# -- Gazdag phase-shift migration on all 8 noisy datasets -----------------------
migrated_gz_noisy = {}

for label, raw in zip(labels_all, outputs_noisy):
    _, shifted = preprocess(raw, dt_ns, t0_ns, time_ns)   # (n_tr, n_t), t0-shifted
    print(f'[{label}] Running Gazdag (noisy) ...', flush=True)
    t0 = _time.perf_counter()
    img = gazdag_migration(
        shifted.T,       # (n_t, n_tr) - time first
        x_traces, time_ns, z_img, v_ice
    )
    print(f'  Done in {_time.perf_counter()-t0:.1f} s  shape={img.shape}')
    migrated_gz_noisy[label] = img


In [ ]:
# -- Plot: full extent ------------------------------------------------------------
fig, axes = plt.subplots(2, 4, figsize=(24, 10), sharex=True, sharey=True)
for i, (ax, (label, img)) in enumerate(zip(axes.ravel(), migrated_gz_noisy.items())):
    vmax = np.max(np.abs(img)) * 0.8
    im = ax.imshow(img, aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax,
              extent=extent_mig, origin='upper')
    ax.plot(x_scatterer_1[0], z_top, 'g*', ms=10, zorder=5, label='x_s1 baseline')
    ax.plot(x_scatterer_1[i], z_top, 'g^', ms=10, zorder=5, label='x_s1 current')
    ax.set_title(f'Gazdag (noisy) - {label}', fontsize=11)
    ax.set_xlabel('x [m]'); ax.set_ylabel('Depth z [m]')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle(
    f'Gazdag Phase-Shift Migration - All 8 Noisy Datasets  |  f_c={f_c_GHz} GHz',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

# -- Plot: zoomed on scatterer region --------------------------------------------
fig, axes = plt.subplots(2, 4, figsize=(24, 10), sharex=True, sharey=True)
for i, (ax, (label, img)) in enumerate(zip(axes.ravel(), migrated_gz_noisy.items())):
    vmax = np.max(np.abs(img)) * 0.8
    im = ax.imshow(img, aspect='auto', cmap='seismic', vmin=-vmax / 20, vmax=vmax / 20,
              extent=extent_mig, origin='upper')
    ax.plot(x_scatterer_1[0], z_top, 'g*', ms=10, zorder=5, label='x_s1 baseline')
    ax.plot(x_scatterer_1[i], z_top, 'g^', ms=10, zorder=5, label='x_s1 current')
    ax.set_xlim(1.5, 2.5); ax.set_ylim(0.75, 0.55)
    ax.set_title(f'{label} (noisy, zoomed)', fontsize=11)
    ax.set_xlabel('x [m]'); ax.set_ylabel('Depth z [m]')
    ax.legend(fontsize=8, loc='upper right')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle(
    f'Gazdag Phase-Shift Migration - Noisy (zoomed)  |  f_c={f_c_GHz} GHz',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()


In [ ]:
# -- Gazdag timelapse differences (noisy migrated - noisy migrated_baseline) ------
migrated_gz_diff_noisy = {label: migrated_gz_noisy[label] - migrated_gz_noisy['Baseline'] for label in labels}

fig, axes = plt.subplots(3, 3, figsize=(18, 10), sharex=True, sharey=True)
axes.ravel()[-2].set_visible(False)
axes.ravel()[-1].set_visible(False)
for i, (ax, (label, diff)) in enumerate(zip(axes.ravel(), migrated_gz_diff_noisy.items())):
    vmax = np.max(np.abs(diff)) * 0.8
    im = ax.imshow(diff, aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax,
              extent=extent_mig, origin='upper')
    ax.plot(x_scatterer_1[0],     z_top, 'g*', ms=10, zorder=5, label='x_s1 baseline')
    ax.plot(x_scatterer_1[i + 1], z_top, 'g^', ms=10, zorder=5, label='x_s1 timelapsed')
    ax.set_title(f'TimeLapse diff (noisy) - {label}', fontsize=11)
    ax.set_xlabel('x [m]'); ax.set_ylabel('Depth z [m]')
    ax.legend(fontsize=8, loc='upper right')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle(
    f'Gazdag Migration - Noisy TimeLapse Differences (migrated - migrated_baseline)  |  f_c={f_c_GHz} GHz',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

# -- Zoomed -------------------------------------------------------------------------
fig, axes = plt.subplots(3, 3, figsize=(18, 10), sharex=True, sharey=True)
axes.ravel()[-2].set_visible(False)
axes.ravel()[-1].set_visible(False)
for i, (ax, (label, diff)) in enumerate(zip(axes.ravel(), migrated_gz_diff_noisy.items())):
    vmax = np.max(np.abs(diff)) * 0.8
    im = ax.imshow(diff, aspect='auto', cmap='seismic', vmin=-vmax / 20, vmax=vmax / 20,
              extent=extent_mig, origin='upper')
    ax.plot(x_scatterer_1[0],     z_top, 'g*', ms=10, zorder=5, label='x_s1 baseline')
    ax.plot(x_scatterer_1[i + 1], z_top, 'g^', ms=10, zorder=5, label='x_s1 timelapsed')
    ax.set_xlim(1.5, 2.5); ax.set_ylim(0.75, 0.55)
    ax.set_title(f'TimeLapse diff (noisy) - {label} (zoomed)', fontsize=11)
    ax.set_xlabel('x [m]'); ax.set_ylabel('Depth z [m]')
    ax.legend(fontsize=8, loc='upper right')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle(
    f'Gazdag Migration - Noisy TimeLapse Differences (zoomed)',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()


## Back-Propagation Migration (Noisy)

Same time-reversal approach as the clean-data section above, but the noisy
background-subtracted traces (`outputs_noisy`) are re-injected as sources instead
of the clean ones. Kirchhoff/Gazdag on noisy data is a pure post-processing step
on already-loaded arrays, but back-propagation requires a brand new gprMax
**forward** (FDTD) simulation per scenario, since the excitation file *is* the
time-reversed noisy trace data.

**Sign-bit time reversal:** spatial focusing during back-propagation is governed
almost entirely by *phase* (zero-crossings), not amplitude — so instead of
injecting the peak-normalised time-reversed wavefield $u(x,\tau)$, we inject only
its sign:

$$u_{\text{sign}}(x, \tau) = \operatorname{sign}\big(u(x, \tau)\big)$$

This keeps every zero-crossing and phase trend of the GPR wavelet intact while
squashing the Laplace noise spikes down to the same $\pm 1$ as the coherent
signal — stripping them of the outsized amplitude that would otherwise let them
dominate the back-propagated wavefield. `write_backprop_files(..., sign_bit=True)`
below implements this (see `helper_functions/migration.py`); the clean-data
section above is left on the default peak-normalised mode since it has no noise
to suppress.

Files are written to `fluidflow_study/backprop/<slug>_noisy/` — slug suffixed
with `_noisy` to keep them alongside, but distinct from, the clean-data runs
already in `backprop/`.

**This notebook only generates the `.in` files below** — run each through gprMax
externally to produce the `.vti` snapshots the load/plot cells expect (the
clean-data section above has the same split: no cell in this notebook calls
`api()` for back-propagation, snapshots are assumed to already exist on disk). E.g.:

```
python -m gprMax fluidflow_study/backprop/2lambda_noisy/backprop_2lambda_noisy.in
```

Run all 8 `.in` files (`baseline_noisy`, `2lambda_noisy`, ..., `0p03125lambda_noisy`),
then re-run the two cells below.


In [ ]:
# -- Back-propagation file generation (noisy) ------------------------------------
# Mirrors the clean-data cell above, but re-injects the noisy background-
# subtracted traces (outputs_noisy) using sign-bit time reversal (sign_bit=True --
# see markdown above) and writes to '<slug>_noisy' folders so the clean-data
# .in/.out/snapshot files are never touched.
slugs_noisy = [f'{slug}_noisy' for slug in slugs_all]

# Sign-bit traces still carry real signal bandwidth that can exceed gprMax's
# numerical-dispersion limit (cells/wavelength) for the finer lambda-fraction
# separations, causing a 'Non-physical wave propagation' error. Low-pass filter
# each excitation file right after writing it so every .in file is runnable.
# eps_r_ice*4 / dx=0.001 mirror the half-velocity material and grid hardcoded in
# write_backprop_files() -- see helper_functions/migration.py.
cutoff_hz = dispersion_limited_cutoff(eps_r=4.0 * eps_r_ice, dx=0.001)

print(f'Back-propagation file generation (noisy, sign-bit)  (stride={STRIDE}, {N_SNAP} snapshots)' + chr(10))
bp_paths_noisy = {}
for label, slug, raw in zip(labels_all, slugs_noisy, outputs_noisy):
    tapered, _ = preprocess(raw, dt_ns, t0_ns, time_ns)
    in_path, n_src, n_snaps, t_focus = write_backprop_files(
        STUDY_ROOT, f'{label} (noisy, sign-bit)', slug, tapered, dt_ns, x_traces,
        t0_ns, eps_r_ice, v_ice, stride=STRIDE, n_snap=N_SNAP, snap_win=SNAP_WIN,
        sign_bit=True
    )
    lowpass_filter_excitation(in_path.parent / 'excitation.txt', cutoff_hz)
    bp_paths_noisy[label] = in_path
    exc_mb = (in_path.parent / 'excitation.txt').stat().st_size / 1e6
    print(f'  [{label}]  {n_src} sources  |  {n_snaps} snapshots  |  '
          f'focus={t_focus:.2f} ns  |  excitation={exc_mb:.1f} MB  |  '
          f'low-pass={cutoff_hz/1e9:.1f} GHz')
    print(f'           {in_path}')

print(chr(10) + 'NOTE: .in files only -- run each through gprMax externally to produce the '
      '.vti snapshots the cells below expect, then re-run them.')


In [ ]:
# -- Sign-bit time-reversed B-scans + their frequency spectra (noisy) -----------
# Recreates the same time-reversal + sign-bit transform used inside
# write_backprop_files(..., sign_bit=True) above, purely for visualisation:
# row 1 shows the resulting B-scan that gets injected into gprMax as the
# source excitation; row 2 shows its frequency spectrum (mean over traces),
# illustrating the broadband content that lowpass_filter_excitation() trims.
sign_bit_bscans = {}
for label, raw in zip(labels_all, outputs_noisy):
    tapered, _ = preprocess(raw, dt_ns, t0_ns, time_ns)   # (n_tr, n_t)
    data_rev = tapered[::STRIDE, ::-1]
    sign_bit_bscans[label] = np.sign(data_rev)            # (n_src, n_t)

freqs_ghz = np.fft.rfftfreq(n_t, d=dt_ns)

fig, axes = plt.subplots(2, 8, figsize=(25, 8))
for ax, (label, sb) in zip(axes[0], sign_bit_bscans.items()):
    ax.imshow(sb.T, aspect='auto', cmap='seismic', extent=extent_bscan,
              interpolation='nearest', vmin=-1, vmax=1)
    ax.set_title(f'{label} (sign-bit)', fontsize=11)
    ax.set_xlabel('x [m]')
axes[0, 0].set_ylabel('Time [ns]')

for ax, (label, sb) in zip(axes[1], sign_bit_bscans.items()):
    spec = np.abs(np.fft.rfft(sb, axis=1)).mean(axis=0)
    ax.plot(freqs_ghz, spec, lw=0.9, color='C2')
    ax.set_title(f'{label} (sign-bit)', fontsize=11)
    ax.set_xlabel('Frequency [GHz]')
axes[1, 0].set_ylabel('|FFT| (mean over traces)')

plt.suptitle('Sign-Bit Time-Reversed Excitation -- B-scans and Spectra (Noisy)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
import pyvista

# -- Load focus frame for all 8 noisy datasets (if snapshots exist) --------------
T_ns_bp    = n_t * dt_ns
t_focus_ns = T_ns_bp - t0_ns
t_start_ns = max(0.0, t_focus_ns - SNAP_WIN)
dt_s       = dt_ns * 1e-9
snap_step_ref = max(1, int((T_ns_bp*1e-9 - t_start_ns*1e-9) / (max(1, N_SNAP - 1) * dt_s)))

focus_frames_noisy = {}
for i, (label, slug) in enumerate(zip(labels_all, slugs_noisy)):
    snap_dir   = STUDY_ROOT / 'backprop' / slug / f'backprop_{slug}_snaps'
    snap_files = sorted(
        snap_dir.glob('bp_snap*.vti'),
        key=lambda p: int(p.stem.replace('bp_snap', ''))
    )
    if not snap_files:
        print(f'[{label}] No snapshots in {snap_dir.name} -- run the .in file through gprMax first, skipping')
        continue

    snap_times_ns = t_start_ns + np.arange(len(snap_files)) * snap_step_ref * dt_ns
    idx_focus     = min(int(np.argmin(np.abs(snap_times_ns - t_focus_ns))),
                        len(snap_files) - 1)

    snaps_mag, snaps_ez = [], []
    for p in snap_files:
        mesh   = pyvista.read(str(p))
        e_data = np.array(mesh['E-field'])
        snaps_mag.append(np.linalg.norm(e_data, axis=1).reshape(1000, 4000))
        snaps_ez.append(e_data[:, 2].reshape(1000, 4000))

    focus_frames_noisy[label] = {
        'mag':      np.stack(snaps_mag)[idx_focus],
        'ez':       np.stack(snaps_ez)[idx_focus],
        't_actual': snap_times_ns[idx_focus],
        'i':        i,
    }
    print(f'[{label}]  {len(snap_files)} snaps  |  focus idx={idx_focus}'
          f'  t={focus_frames_noisy[label]["t_actual"]:.3f} ns')

if not focus_frames_noisy:
    print('\nNo noisy back-propagation snapshots found yet -- run the .in files '
          'generated above through gprMax, then re-run this cell.')
else:
    extent_full = [0, 4.0, 0, 1]
    margin_x    = 0.4
    margin_y    = 0.12
    x_zoom_lo = x_scatterer_1[0] - margin_x
    x_zoom_hi = x_scatterer_1[1] + margin_x

    def _add_markers_noisy(ax, i):
        ax.plot(x_scatterer_1[0], y_scatterer, 'g*', ms=10, zorder=5, label='x_s1 baseline')
        ax.plot(x_scatterer_1[i], y_scatterer, 'g^', ms=10, zorder=5, label='x_s1 current')

    # |E| magnitude -- full extent 2x4 grid
    fig, axes = plt.subplots(2, 4, figsize=(24, 10), sharex=True, sharey=True)
    for ax, (label, frame) in zip(axes.ravel(), focus_frames_noisy.items()):
        ax.imshow(frame['mag'], aspect='auto', cmap='inferno',
                  extent=extent_full, origin='lower')
        _add_markers_noisy(ax, frame['i'])
        ax.axhline(y_surface, color='cyan', lw=0.8, ls='--')
        ax.set_title(f'{label} (noisy)  |  t={frame["t_actual"]:.2f} ns', fontsize=11)
        ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
    axes.ravel()[0].legend(fontsize=8, loc='upper right')
    plt.suptitle(f'Back-Propagation |E| (Noisy) -- All Datasets  |  focus at {t_focus_ns:.2f} ns',
                 fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

    # |E| magnitude -- zoomed 2x4 grid
    fig, axes = plt.subplots(2, 4, figsize=(24, 10), sharex=True, sharey=True)
    for ax, (label, frame) in zip(axes.ravel(), focus_frames_noisy.items()):
        ax.imshow(frame['mag'], aspect='auto', cmap='inferno',
                  extent=extent_full, origin='lower')
        _add_markers_noisy(ax, frame['i'])
        ax.set_xlim(x_zoom_lo, x_zoom_hi)
        ax.set_ylim(y_scatterer - margin_y, y_scatterer + margin_y)
        ax.set_title(f'{label} (noisy, zoomed)', fontsize=11)
        ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
        ax.legend(fontsize=8, loc='upper right')
    plt.suptitle(f'Back-Propagation |E| (Noisy, zoomed)  |  focus at {t_focus_ns:.2f} ns',
                 fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

    # Ez component -- full extent 2x4 grid
    fig, axes = plt.subplots(2, 4, figsize=(24, 10), sharex=True, sharey=True)
    for ax, (label, frame) in zip(axes.ravel(), focus_frames_noisy.items()):
        ax.imshow(frame['ez'], aspect='auto', cmap='seismic',
                  extent=extent_full, origin='lower')
        _add_markers_noisy(ax, frame['i'])
        ax.axhline(y_surface, color='cyan', lw=0.8, ls='--')
        ax.set_title(f'{label} (noisy)  |  t={frame["t_actual"]:.2f} ns', fontsize=11)
        ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
    axes.ravel()[0].legend(fontsize=8, loc='upper right')
    plt.suptitle(f'Back-Propagation Ez (Noisy) -- All Datasets  |  focus at {t_focus_ns:.2f} ns',
                 fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

    # Ez component -- zoomed 2x4 grid
    fig, axes = plt.subplots(2, 4, figsize=(24, 10), sharex=True, sharey=True)
    for ax, (label, frame) in zip(axes.ravel(), focus_frames_noisy.items()):
        ax.imshow(frame['ez'], aspect='auto', cmap='seismic',
                  extent=extent_full, origin='lower')
        _add_markers_noisy(ax, frame['i'])
        ax.set_xlim(x_zoom_lo, x_zoom_hi)
        ax.set_ylim(y_scatterer - margin_y, y_scatterer + margin_y)
        ax.set_title(f'{label} (noisy, zoomed)', fontsize=11)
        ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
        ax.legend(fontsize=8, loc='upper right')
    plt.suptitle(f'Back-Propagation Ez (Noisy, zoomed)  |  focus at {t_focus_ns:.2f} ns',
                 fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()


In [ ]:
# -- Back-propagation timelapse differences (noisy) (Ez_noisy - Ez_noisy_baseline) --
if not focus_frames_noisy or 'Baseline' not in focus_frames_noisy:
    print('Skipping noisy back-propagation timelapse-difference plot -- '
          'no noisy snapshots (or no Baseline snapshot) yet.')
else:
    bp_ez_diff_noisy = {label: focus_frames_noisy[label]['ez'] - focus_frames_noisy['Baseline']['ez']
                         for label in labels if label in focus_frames_noisy}

    fig, axes = plt.subplots(2, 4, figsize=(18, 10), sharex=True, sharey=True)
    axes.ravel()[-1].set_visible(False)
    for i, (ax, (label, diff)) in enumerate(zip(axes.ravel(), bp_ez_diff_noisy.items())):
        vmax = np.percentile(np.abs(diff), 100)
        ax.imshow(diff, aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax,
                  extent=extent_full, origin='lower')
        ax.plot(x_scatterer_1[0],     y_scatterer, 'g*', ms=10, zorder=5, label='x_s1 baseline')
        ax.plot(x_scatterer_1[i + 1], y_scatterer, 'g^', ms=10, zorder=5, label='x_s1 timelapsed')
        ax.set_title(f'TimeLapse diff Ez (noisy) -- {label}', fontsize=11)
        ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
        ax.legend(fontsize=8, loc='upper right')
    plt.suptitle(
        f'Back-Propagation (Noisy) -- TimeLapse Differences Ez  |  focus at {t_focus_ns:.2f} ns',
        fontsize=12, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    plt.show()

    # Zoomed
    fig, axes = plt.subplots(2, 4, figsize=(18, 10), sharex=True, sharey=True)
    axes.ravel()[-1].set_visible(False)
    for i, (ax, (label, diff)) in enumerate(zip(axes.ravel(), bp_ez_diff_noisy.items())):
        vmax = np.percentile(np.abs(diff), 100)
        ax.imshow(diff, aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax,
                  extent=extent_full, origin='lower')
        ax.plot(x_scatterer_1[0],     y_scatterer, 'g*', ms=10, zorder=5, label='x_s1 baseline')
        ax.plot(x_scatterer_1[i + 1], y_scatterer, 'g^', ms=10, zorder=5, label='x_s1 timelapsed')
        ax.set_xlim(x_zoom_lo, x_zoom_hi)
        ax.set_ylim(y_scatterer - margin_y, y_scatterer + margin_y)
        ax.set_title(f'TimeLapse diff Ez (noisy) -- {label} (zoomed)', fontsize=11)
        ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
        ax.legend(fontsize=8, loc='upper right')
    plt.suptitle('Back-Propagation (Noisy) -- TimeLapse Differences Ez (zoomed)',
                 fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()


## Save Migrated Results (Noisy, non-difference)

In [ ]:
# -- Save normal (non-difference) migrated results from the noisy datasets -----
# Stacks all 8 scenarios (Baseline + 7 shifts) from migrated_noisy, migrated_gz_noisy,
# and focus_frames_noisy (back-prop) into migrated_results_noisy.npz.
# Back-propagation is only included for scenarios whose .in file (generated above)
# has actually been run through gprMax externally and produced snapshots that the
# noisy-data loading cell above found (focus_frames_noisy) -- otherwise that
# scenario's plane is filled with NaN, same convention as migrated_results.npz.

kirchhoff_noisy_all = np.stack([migrated_noisy[lbl].astype(float)    for lbl in labels_all], axis=0)  # (8, n_z, n_x)
gazdag_noisy_all    = np.stack([migrated_gz_noisy[lbl].astype(float) for lbl in labels_all], axis=0)

# -- Back-prop (noisy): reproject focus_frames_noisy ez onto migration grid -----
x_ax_bp = np.linspace(0, 4.0, 4000)
y_ax_bp = np.linspace(0, 1.0, 1000)
y_mig   = y_surface - z_img
Yq, Xq  = np.meshgrid(y_mig, x_traces, indexing='ij')

from scipy.interpolate import RegularGridInterpolator as _RGI
backprop_noisy_planes = []
for lbl in labels_all:
    if lbl in focus_frames_noisy:
        interp = _RGI((y_ax_bp, x_ax_bp), focus_frames_noisy[lbl]['ez'],
                      method='linear', bounds_error=False, fill_value=0.0)
        backprop_noisy_planes.append(interp((Yq, Xq)).astype(float))
    else:
        backprop_noisy_planes.append(np.full((len(z_img), len(x_traces)), np.nan))
backprop_noisy_all = np.stack(backprop_noisy_planes, axis=0)  # (8, n_z, n_x)

save_path_mig_noisy = STUDY_ROOT / "migrated_results_noisy.npz"

np.savez_compressed(
    save_path_mig_noisy,
    kirchhoff         = kirchhoff_noisy_all,
    gazdag            = gazdag_noisy_all,
    backprop          = backprop_noisy_all,
    scenarios         = np.array(labels_all, dtype="U20"),               # (8,)  Baseline + 7 shifts
    separation_lambda = np.array([0, 2, 1, 0.5, 0.25, 0.125, 0.0625, 0.03125]),  # (8,)  0 = baseline
    x_traces          = x_traces,
    z_img             = z_img,
    x_scatterer       = np.array(x_scatterer_1),
    z_scatterer       = np.float64(z_scatterer),
    z_top             = np.float64(z_top),
    noise_level       = np.float64(NOISE_LEVEL),
)

print(f"Saved -> {save_path_mig_noisy}")
print(f"\nStacked arrays  (n_scenarios=8, n_z={kirchhoff_noisy_all.shape[1]}, n_x={kirchhoff_noisy_all.shape[2]}):")
for name, arr in [("kirchhoff", kirchhoff_noisy_all),
                  ("gazdag",    gazdag_noisy_all),
                  ("backprop",  backprop_noisy_all)]:
    absent = int(np.isnan(arr).all(axis=(1, 2)).sum())
    note   = f"  ({absent} scenario(s) absent -> NaN)" if absent else ""
    print(f"  {name:<12}  {arr.shape}{note}")
print(f"\nUsage examples:")
print(f"  d = np.load(str(STUDY_ROOT / 'migrated_results_noisy.npz'), allow_pickle=False)")
print(f"  d['kirchhoff'][0]   # Kirchhoff Baseline (noisy) -> shape (n_z, n_x)")
print(f"  d['gazdag'][3]      # Gazdag 1/2 lambda (noisy)  -> shape (n_z, n_x)")
print(f"  d['backprop'][1]    # Back-prop 2 lambda (noisy) -> shape (n_z, n_x), NaN if not yet run")
